# Kinetiq ERP/WMS: SYNTHETIC data generator and loader for the private Azure SQL database

Creates the `ref`, `erp` and `wms` schemas in the ERP/WMS Azure SQL database and loads **SYNTHETIC** data. The
generator code is **in this notebook** (one cell per table, no package, no `%pip install`). It shares the world model
with `01_generate_batch_sources` (same code, same seed), so stock levels match the Databricks sell-through history.

**Why this runs in Databricks:** the SQL server has public network access disabled (enforced by tenant policy).
This VNet-injected cluster reaches it through the SQL private endpoint (`privatelink.database.windows.net` zone linked
to the VNet). Authentication uses the Unity Catalog **service credential** backed by the Databricks access connector's
managed identity (a member of the SQL Entra admin group), so no passwords or secrets are involved.

**Modes:** `write` deletes and re-inserts the synthetic tables (the write-back tables `erp.allocation_decision`,
`erp.stock_transfer_order` and the agent tables are created if missing and never truncated). `preview` generates and
shows the tables without connecting to SQL.

**Row counts:** master data follows the dimensions (markets, stores per market, styles). Transaction volumes have
their own widgets (orders per partner, purchase-order coverage, movement days and movements per day).

In [ ]:
dbutils.widgets.dropdown("mode", "write", ["write", "preview"], "0. mode")
dbutils.widgets.dropdown("profile", "demo", ["tiny", "small", "demo", "large"], "1. size profile")
dbutils.widgets.text("seed", "", "2. seed (blank = 20261106)")
dbutils.widgets.text("as_of", "", "3. as-of date (blank = 2026-10-31)")
dbutils.widgets.text("n_markets", "", "4. markets 5-10 (override)")
dbutils.widgets.text("stores_per_market", "", "5. stores per market 3-60 (override)")
dbutils.widgets.text("n_styles", "", "6. styles 12-200 (override)")
dbutils.widgets.text("orders_per_key_partner", "3", "7. wholesale orders per KEY partner")
dbutils.widgets.text("orders_per_partner", "2", "8. wholesale orders per other partner")
dbutils.widgets.text("po_style_share", "0.35", "9. share of styles x DCs with an open PO (0-1)")
dbutils.widgets.text("movement_days", "14", "a. days of WMS movement history")
dbutils.widgets.text("movements_per_day", "60", "b. replenishment movements per day")
dbutils.widgets.text("sql_server", "", "c. SQL server FQDN")
dbutils.widgets.text("sql_database", "sqldb-appareliq-erpwms", "d. SQL database")
dbutils.widgets.text("service_credential", "appareliq-sql", "e. UC service credential")


## Shared world model (identical in notebooks 01 and 02)

The next four cells define the fictional Kinetiq world: static reference lists, parameters, master data
(markets, stores, DCs, styles, SKUs, assortment, prices, partners, lanes, people, events, scenario) and the
expected-demand model. Notebook `02_load_erp_wms_sql` contains **the same code**, so with the same parameters both
notebooks agree on every key and the ERP/WMS stock is consistent with the Databricks sell-through history.
Keep the two copies in sync when you change anything here.

In [ ]:
# Reference data: static lists. Cities are real places; every brand, product, store, partner, vendor
# and person name is invented.
import hashlib
import json
import math
import time
import zlib
from dataclasses import asdict, dataclass, field, fields
from datetime import date, datetime, timedelta, timezone
from datetime import time as dtime

import numpy as np
import pandas as pd


# market_id, name, region, currency, usd->local fx, price premium vs US, hemisphere, timezone, price rounding
MARKETS = [
    ("US", "United States", "NORTH_AMERICA", "USD", 1.00, 1.00, "N", "America/New_York", "x.99"),
    ("GB", "United Kingdom", "EUROPE", "GBP", 0.79, 0.98, "N", "Europe/London", "x.99"),
    ("JP", "Japan", "ASIA", "JPY", 150.0, 1.06, "N", "Asia/Tokyo", "100"),
    ("BR", "Brazil", "LATAM", "BRL", 5.05, 1.45, "S", "America/Sao_Paulo", "x.99"),
    ("AU", "Australia", "APAC", "AUD", 1.52, 1.10, "S", "Australia/Sydney", "x.99"),
    ("DE", "Germany", "EUROPE", "EUR", 0.92, 1.05, "N", "Europe/Berlin", "x.99"),
    ("FR", "France", "EUROPE", "EUR", 0.92, 1.05, "N", "Europe/Paris", "x.99"),
    ("CA", "Canada", "NORTH_AMERICA", "CAD", 1.36, 1.02, "N", "America/Toronto", "x.99"),
    ("CN", "China", "GREATER_CHINA", "CNY", 7.20, 1.00, "N", "Asia/Shanghai", "x9"),
    ("KR", "South Korea", "ASIA", "KRW", 1350.0, 1.02, "N", "Asia/Seoul", "1000"),
]

# market -> [(city, state/province, lat, lon, timezone, [streets/districts])]; first city hosts the flagship
CITIES = {
    "US": [("New York", "NY", 40.7580, -73.9855, "America/New_York", ["Fifth Avenue", "SoHo", "Brooklyn Williamsburg"]),
           ("Los Angeles", "CA", 34.0522, -118.2437, "America/Los_Angeles", ["The Grove", "Melrose", "Santa Monica"]),
           ("Chicago", "IL", 41.8781, -87.6298, "America/Chicago", ["Michigan Avenue", "Wicker Park"]),
           ("Houston", "TX", 29.7604, -95.3698, "America/Chicago", ["Galleria", "Midtown"]),
           ("Atlanta", "GA", 33.7490, -84.3880, "America/New_York", ["Lenox", "West End"]),
           ("Seattle", "WA", 47.6062, -122.3321, "America/Los_Angeles", ["Pike Street", "University Village"]),
           ("Boston", "MA", 42.3601, -71.0589, "America/New_York", ["Newbury Street", "Seaport"]),
           ("Miami", "FL", 25.7617, -80.1918, "America/New_York", ["Lincoln Road", "Brickell"]),
           ("Denver", "CO", 39.7392, -104.9903, "America/Denver", ["Cherry Creek", "LoDo"]),
           ("Philadelphia", "PA", 39.9526, -75.1652, "America/New_York", ["Walnut Street", "Center City"]),
           ("Portland", "OR", 45.5152, -122.6784, "America/Los_Angeles", ["Pearl District"]),
           ("Dallas", "TX", 32.7767, -96.7970, "America/Chicago", ["NorthPark", "Deep Ellum"])],
    "GB": [("London", "Greater London", 51.5145, -0.1440, "Europe/London", ["Oxford Street", "Covent Garden", "Shoreditch", "Westfield Stratford"]),
           ("Manchester", "Greater Manchester", 53.4808, -2.2426, "Europe/London", ["Market Street", "Trafford"]),
           ("Birmingham", "West Midlands", 52.4862, -1.8904, "Europe/London", ["Bullring"]),
           ("Glasgow", "Scotland", 55.8642, -4.2518, "Europe/London", ["Buchanan Street"]),
           ("Leeds", "West Yorkshire", 53.8008, -1.5491, "Europe/London", ["Briggate"]),
           ("Liverpool", "Merseyside", 53.4084, -2.9916, "Europe/London", ["Church Street"]),
           ("Bristol", "Bristol", 51.4545, -2.5879, "Europe/London", ["Cabot Circus"]),
           ("Edinburgh", "Scotland", 55.9533, -3.1883, "Europe/London", ["Princes Street"])],
    "JP": [("Tokyo", "Tokyo", 35.6702, 139.7027, "Asia/Tokyo", ["Harajuku", "Shibuya", "Ginza", "Shinjuku"]),
           ("Osaka", "Osaka", 34.6937, 135.5023, "Asia/Tokyo", ["Shinsaibashi", "Umeda"]),
           ("Yokohama", "Kanagawa", 35.4437, 139.6380, "Asia/Tokyo", ["Minato Mirai"]),
           ("Nagoya", "Aichi", 35.1815, 136.9066, "Asia/Tokyo", ["Sakae"]),
           ("Fukuoka", "Fukuoka", 33.5904, 130.4017, "Asia/Tokyo", ["Tenjin"]),
           ("Sapporo", "Hokkaido", 43.0618, 141.3545, "Asia/Tokyo", ["Odori"]),
           ("Kyoto", "Kyoto", 35.0116, 135.7681, "Asia/Tokyo", ["Shijo"])],
    "BR": [("São Paulo", "SP", -23.5614, -46.6559, "America/Sao_Paulo", ["Avenida Paulista", "Rua Oscar Freire", "Pinheiros"]),
           ("Rio de Janeiro", "RJ", -22.9068, -43.1729, "America/Sao_Paulo", ["Ipanema", "Barra da Tijuca"]),
           ("Belo Horizonte", "MG", -19.9167, -43.9345, "America/Sao_Paulo", ["Savassi"]),
           ("Brasília", "DF", -15.7975, -47.8919, "America/Sao_Paulo", ["Asa Sul"]),
           ("Curitiba", "PR", -25.4284, -49.2733, "America/Sao_Paulo", ["Batel"]),
           ("Porto Alegre", "RS", -30.0346, -51.2177, "America/Sao_Paulo", ["Moinhos de Vento"]),
           ("Recife", "PE", -8.0476, -34.8770, "America/Recife", ["Boa Viagem"])],
    "AU": [("Sydney", "NSW", -33.8688, 151.2093, "Australia/Sydney", ["Pitt Street", "Bondi Junction", "Parramatta"]),
           ("Melbourne", "VIC", -37.8136, 144.9631, "Australia/Melbourne", ["Bourke Street", "Chadstone"]),
           ("Brisbane", "QLD", -27.4698, 153.0251, "Australia/Brisbane", ["Queen Street"]),
           ("Perth", "WA", -31.9505, 115.8605, "Australia/Perth", ["Murray Street"]),
           ("Adelaide", "SA", -34.9285, 138.6007, "Australia/Adelaide", ["Rundle Mall"]),
           ("Gold Coast", "QLD", -28.0167, 153.4000, "Australia/Brisbane", ["Pacific Fair"])],
    "DE": [("Berlin", "Berlin", 52.5200, 13.4050, "Europe/Berlin", ["Kurfürstendamm", "Alexanderplatz", "Mitte"]),
           ("Munich", "Bavaria", 48.1351, 11.5820, "Europe/Berlin", ["Kaufingerstraße"]),
           ("Hamburg", "Hamburg", 53.5511, 9.9937, "Europe/Berlin", ["Mönckebergstraße", "Schanzenviertel"]),
           ("Frankfurt", "Hesse", 50.1109, 8.6821, "Europe/Berlin", ["Zeil"]),
           ("Cologne", "North Rhine-Westphalia", 50.9375, 6.9603, "Europe/Berlin", ["Schildergasse"]),
           ("Düsseldorf", "North Rhine-Westphalia", 51.2277, 6.7735, "Europe/Berlin", ["Königsallee"])],
    "FR": [("Paris", "Île-de-France", 48.8698, 2.3075, "Europe/Paris", ["Champs-Élysées", "Le Marais", "Les Halles"]),
           ("Lyon", "Auvergne-Rhône-Alpes", 45.7640, 4.8357, "Europe/Paris", ["Rue de la République"]),
           ("Marseille", "Provence-Alpes-Côte d'Azur", 43.2965, 5.3698, "Europe/Paris", ["Terrasses du Port"]),
           ("Bordeaux", "Nouvelle-Aquitaine", 44.8378, -0.5792, "Europe/Paris", ["Rue Sainte-Catherine"]),
           ("Lille", "Hauts-de-France", 50.6292, 3.0573, "Europe/Paris", ["Euralille"]),
           ("Toulouse", "Occitanie", 43.6047, 1.4442, "Europe/Paris", ["Capitole"])],
    "CA": [("Toronto", "ON", 43.6532, -79.3832, "America/Toronto", ["Eaton Centre", "Queen Street West", "Yorkdale"]),
           ("Vancouver", "BC", 49.2827, -123.1207, "America/Vancouver", ["Robson Street", "Metrotown"]),
           ("Montreal", "QC", 45.5019, -73.5674, "America/Toronto", ["Rue Sainte-Catherine"]),
           ("Calgary", "AB", 51.0447, -114.0719, "America/Edmonton", ["Chinook Centre"]),
           ("Ottawa", "ON", 45.4215, -75.6972, "America/Toronto", ["Rideau Centre"])],
    "CN": [("Shanghai", "Shanghai", 31.2304, 121.4737, "Asia/Shanghai", ["Nanjing Road", "Huaihai Road", "Xintiandi"]),
           ("Beijing", "Beijing", 39.9042, 116.4074, "Asia/Shanghai", ["Sanlitun", "Wangfujing"]),
           ("Guangzhou", "Guangdong", 23.1291, 113.2644, "Asia/Shanghai", ["Tianhe"]),
           ("Shenzhen", "Guangdong", 22.5431, 114.0579, "Asia/Shanghai", ["Futian"]),
           ("Chengdu", "Sichuan", 30.5728, 104.0668, "Asia/Shanghai", ["Chunxi Road"]),
           ("Hangzhou", "Zhejiang", 30.2741, 120.1551, "Asia/Shanghai", ["Hubin"])],
    "KR": [("Seoul", "Seoul", 37.5563, 126.9236, "Asia/Seoul", ["Myeongdong", "Hongdae", "Gangnam"]),
           ("Busan", "Busan", 35.1796, 129.0756, "Asia/Seoul", ["Seomyeon", "Haeundae"]),
           ("Incheon", "Incheon", 37.4563, 126.7052, "Asia/Seoul", ["Songdo"]),
           ("Daegu", "Daegu", 35.8714, 128.6014, "Asia/Seoul", ["Dongseongno"]),
           ("Daejeon", "Daejeon", 36.3504, 127.3845, "Asia/Seoul", ["Dunsan"])],
}

# Outlet towns for factory stores: market -> [(town, state, lat, lon, tz)]
OUTLET_TOWNS = {
    "US": [("Orlando", "FL", 28.4446, -81.4501, "America/New_York"), ("Woodbury", "NY", 41.3134, -74.1182, "America/New_York"),
           ("Cabazon", "CA", 33.9175, -116.7870, "America/Los_Angeles"), ("San Marcos", "TX", 29.8278, -97.9817, "America/Chicago")],
    "GB": [("Bicester", "Oxfordshire", 51.8916, -1.1534, "Europe/London"), ("Cheshire Oaks", "Cheshire", 53.2667, -2.8833, "Europe/London")],
    "JP": [("Gotemba", "Shizuoka", 35.3086, 138.9653, "Asia/Tokyo"), ("Rinku", "Osaka", 34.4097, 135.2967, "Asia/Tokyo")],
    "BR": [("Itupeva", "SP", -23.1531, -47.0578, "America/Sao_Paulo")],
    "AU": [("Essendon Fields", "VIC", -37.7286, 144.9026, "Australia/Melbourne"), ("Homebush", "NSW", -33.8650, 151.0800, "Australia/Sydney")],
    "DE": [("Metzingen", "Baden-Württemberg", 48.5367, 9.2858, "Europe/Berlin"), ("Wustermark", "Brandenburg", 52.5500, 12.9500, "Europe/Berlin")],
    "FR": [("Serris", "Île-de-France", 48.8540, 2.7840, "Europe/Paris"), ("Roubaix", "Hauts-de-France", 50.6942, 3.1746, "Europe/Paris")],
    "CA": [("Halton Hills", "ON", 43.6300, -79.9500, "America/Toronto")],
    "CN": [("Kunshan", "Jiangsu", 31.3858, 120.9806, "Asia/Shanghai")],
    "KR": [("Yeoju", "Gyeonggi", 37.2983, 127.6375, "Asia/Seoul"), ("Gimhae", "Gyeongnam", 35.2285, 128.8894, "Asia/Seoul")],
}

# Distribution centres: (dc_id, market, city, state, lat, lon, tz, priority); priority 1 = primary DC for the market
DCS = [
    ("DC-US-MEM", "US", "Memphis", "TN", 35.1495, -90.0490, "America/Chicago", 1),
    ("DC-GB-DAV", "GB", "Daventry", "Northamptonshire", 52.2565, -1.1626, "Europe/London", 1),
    ("DC-JP-ICH", "JP", "Ichikawa", "Chiba", 35.7219, 139.9310, "Asia/Tokyo", 1),
    ("DC-BR-CAJ", "BR", "Cajamar", "SP", -23.3550, -46.8769, "America/Sao_Paulo", 1),
    ("DC-AU-TRU", "AU", "Truganina", "VIC", -37.8167, 144.7333, "Australia/Melbourne", 1),
    ("DC-DE-ERF", "DE", "Erfurt", "Thuringia", 50.9848, 11.0299, "Europe/Berlin", 1),
    ("DC-FR-LIE", "FR", "Lieusaint", "Île-de-France", 48.6333, 2.5500, "Europe/Paris", 1),
    ("DC-CA-MIS", "CA", "Mississauga", "ON", 43.5890, -79.6441, "America/Toronto", 1),
    ("DC-CN-TAI", "CN", "Taicang", "Jiangsu", 31.4597, 121.1302, "Asia/Shanghai", 1),
    ("DC-KR-ICN", "KR", "Incheon", "Incheon", 37.4563, 126.7052, "Asia/Seoul", 1),
    ("DC-US-ONT", "US", "Ontario", "CA", 34.0633, -117.6509, "America/Los_Angeles", 2),
    ("DC-US-ALT", "US", "Allentown", "PA", 40.6084, -75.4902, "America/New_York", 2),
]

# Product families: (category, division, gender, base model names, msrp tiers USD, size run, popularity)
FAMILIES = [
    ("RUNNING", "FOOTWEAR", "M", ["AeroGlide", "Pacer", "Velo Stride", "Tempo Flux", "Cloudline"], [110, 130, 160, 180], "MENS_FW", 1.30),
    ("RUNNING", "FOOTWEAR", "W", ["AeroGlide", "Pacer", "Velo Stride", "Tempo Flux", "Cloudline"], [110, 130, 160, 180], "WOMENS_FW", 1.15),
    ("TRAIL", "FOOTWEAR", "M", ["Ridgeline", "Summit Trek", "Scree Runner"], [120, 140, 165], "MENS_FW", 0.55),
    ("TRAIL", "FOOTWEAR", "W", ["Ridgeline", "Summit Trek"], [120, 140], "WOMENS_FW", 0.45),
    ("BASKETBALL", "FOOTWEAR", "M", ["Court Vortex", "Skyrise", "Rim Elite", "Baseline Pro"], [100, 125, 150, 190], "MENS_FW", 0.95),
    ("LIFESTYLE", "FOOTWEAR", "M", ["Heritage 82", "Metro Classic", "Cortex Low", "Retro Runner"], [85, 100, 115, 130], "MENS_FW", 1.25),
    ("LIFESTYLE", "FOOTWEAR", "W", ["Heritage 82", "Metro Classic", "Cortex Low", "Platform Muse"], [85, 100, 115, 130], "WOMENS_FW", 1.20),
    ("TRAINING", "FOOTWEAR", "M", ["Flexform TR", "Grit Trainer", "Lift Core"], [90, 110, 130], "MENS_FW", 0.60),
    ("TRAINING", "FOOTWEAR", "W", ["Flexform TR", "Grit Trainer", "Studio Flow"], [85, 100, 120], "WOMENS_FW", 0.65),
    ("FOOTBALL", "FOOTWEAR", "M", ["Strike Elite FG", "Apex Control FG", "Velocity Club FG"], [70, 150, 260], "MENS_FW", 0.50),
    ("APPAREL", "APPAREL", "M", ["AirLite Run Tee", "StormShield Jacket", "CoreFleece Hoodie", "Stride Short 7in", "Tempo Tight"], [35, 45, 70, 90, 130], "APPAREL", 0.90),
    ("APPAREL", "APPAREL", "W", ["AirLite Run Tank", "StormShield Jacket", "CoreFleece Crew", "Stride Short 3in", "Sculpt Legging"], [35, 45, 70, 80, 110], "APPAREL", 0.95),
]

STYLE_PREFIXES = ["HV", "FV", "DV", "FD", "DX", "HQ", "FJ", "IB", "FN", "HF"]

COLORWAYS = [
    ("001", "Black/White"), ("100", "White/Black"), ("002", "Black/Anthracite-Cool Grey"), ("101", "White/Summit White-Pure Platinum"),
    ("400", "Midnight Navy/White"), ("600", "University Red/Black"), ("300", "Volt/Black"), ("700", "Laser Orange/Sail"),
    ("401", "Racer Blue/White-Black"), ("500", "Hyper Violet/Pink Blast"), ("200", "Light Bone/Sail-Gum"), ("003", "Iron Grey/Smoke Grey"),
    ("301", "Electric Green/Deep Jungle"), ("601", "Team Crimson/Phantom"), ("402", "Glacier Blue/Ice"), ("201", "Khaki/Sequoia"),
]

SIZE_RUNS = {
    "MENS_FW": ["6", "6.5", "7", "7.5", "8", "8.5", "9", "9.5", "10", "10.5", "11", "11.5", "12", "13", "14"],
    "WOMENS_FW": ["5", "5.5", "6", "6.5", "7", "7.5", "8", "8.5", "9", "9.5", "10", "11"],
    "APPAREL": ["XS", "S", "M", "L", "XL", "XXL"],
}
SIZE_CURVE_CENTER = {"MENS_FW": (10.0, 1.6), "WOMENS_FW": (7.75, 1.4)}
APPAREL_CURVE = {"XS": 0.06, "S": 0.18, "M": 0.30, "L": 0.26, "XL": 0.14, "XXL": 0.06}

RETURN_RATE = {"FOOTWEAR": 0.055, "APPAREL": 0.085}
UNIT_COST_RATIO = {"FOOTWEAR": 0.36, "APPAREL": 0.30}

WHOLESALE_PARTNERS = [
    ("Summit Sports Group", "US", "KEY"), ("Swiftline Athletics", "US", "KEY"), ("Courtside Collective", "US", "CORE"),
    ("Harbor & Field Sports", "GB", "KEY"), ("Northline Outfitters", "GB", "CORE"), ("Stride & Co", "GB", "CORE"),
    ("Sakura Run Depot", "JP", "KEY"), ("Kanto Athletic House", "JP", "CORE"), ("Tropico Esportes", "BR", "KEY"),
    ("Arena Sul Esportes", "BR", "CORE"), ("Southern Cross Sport", "AU", "KEY"), ("Coastline Athletic", "AU", "CORE"),
    ("Rhein Sporthaus", "DE", "KEY"), ("Alpenlauf Sport", "DE", "CORE"), ("Maison du Sport Lumière", "FR", "KEY"),
    ("Maple Leaf Athletics", "CA", "KEY"), ("Jade River Sports", "CN", "KEY"), ("Hanbit Sports", "KR", "KEY"),
]

VENDORS = [
    ("VN-DNF", "Dong Nai Footwear Co. (synthetic vendor)", "VN", 38),
    ("VN-BDS", "Binh Duong Shoe Works (synthetic vendor)", "VN", 40),
    ("ID-TGR", "Tangerang Footwear Industries (synthetic vendor)", "ID", 42),
    ("CN-PUT", "Putian Athletic Manufacturing (synthetic vendor)", "CN", 30),
    ("KH-PPA", "Phnom Penh Apparel Ltd. (synthetic vendor)", "KH", 45),
    ("BD-DHA", "Dhaka Knit Partners (synthetic vendor)", "BD", 48),
]

# People (all invented). (person_id, display name, title, market or GLOBAL)
PEOPLE = [
    ("dortiz", "Daniela Ortiz", "Regional Marketplace Director, North America", "US"),
    ("pnair", "Priya Nair", "Senior Merchandise Planner, Running", "US"),
    ("mlee", "Marcus Lee", "Wholesale Account Lead, North America", "US"),
    ("ohaddad", "Omar Haddad", "Finance Partner, Marketplace", "GLOBAL"),
    ("fbrandt", "Fiona Brandt", "Marketplace Director, Europe", "GB"),
    ("lweber", "Lukas Weber", "Merchandise Planner, Europe", "DE"),
    ("erossi", "Elena Rossi", "Wholesale Account Lead, Europe", "GB"),
    ("mtan", "Mei Tan", "Marketplace Director, Asia Pacific", "JP"),
    ("asato", "Aiko Sato", "Merchandise Planner, Japan", "JP"),
    ("gkim", "Grace Kim", "Inventory Analyst, Asia Pacific", "AU"),
    ("slima", "Sofia Lima", "Merchandise Planner, Brazil", "BR"),
    ("lcarter", "Liam Carter", "Merchandise Planner, Australia", "AU"),
    ("jmartin", "Julien Martin", "Merchandise Planner, France", "FR"),
    ("cwong", "Chloe Wong", "Merchandise Planner, Canada", "CA"),
    ("hzhang", "Hao Zhang", "Merchandise Planner, Greater China", "CN"),
    ("jpark", "Jiwoo Park", "Merchandise Planner, Korea", "KR"),
]
# Invented store-manager names (one manager per store; picked deterministically from the seed)
STORE_MANAGER_FIRST = ["Alex", "Jordan", "Sam", "Taylor", "Morgan", "Casey", "Riley", "Jamie", "Avery", "Quinn",
                       "Rowan", "Harper", "Emery", "Dakota", "Reese", "Skyler", "Parker", "Sage", "Remy", "Noa"]
STORE_MANAGER_LAST = ["Bennett", "Okafor", "Silva", "Novak", "Ramos", "Lindqvist", "Moreau", "Tanaka", "Haddad",
                      "Kowalski", "Fischer", "Mensah", "Duarte", "Ivanova", "Park", "Rossi", "Nakamura", "Byrne"]
MARKET_DIRECTOR = {"US": "dortiz", "CA": "dortiz", "GB": "fbrandt", "DE": "fbrandt", "FR": "fbrandt",
                   "JP": "mtan", "AU": "mtan", "CN": "mtan", "KR": "mtan", "BR": "dortiz"}
MARKET_PLANNER = {"US": "pnair", "GB": "lweber", "DE": "lweber", "FR": "jmartin", "JP": "asato", "BR": "slima",
                  "AU": "lcarter", "CA": "cwong", "CN": "hzhang", "KR": "jpark"}
WHOLESALE_LEAD = {"US": "mlee", "CA": "mlee", "BR": "mlee", "GB": "erossi", "DE": "erossi", "FR": "erossi",
                  "JP": "gkim", "AU": "gkim", "CN": "gkim", "KR": "gkim"}

# Regional events that drive demand spikes: (market, name, month, day, duration_days, categories, uplift)
REGIONAL_EVENTS = [
    ("US", "Big Apple City Marathon Expo", 11, 1, 3, ["RUNNING"], 1.6),
    ("US", "Back-to-School Season", 8, 10, 21, ["LIFESTYLE", "BASKETBALL", "APPAREL"], 1.25),
    ("US", "Black Friday Weekend", 11, 27, 4, None, 1.9),
    ("GB", "Thames Half Marathon", 10, 12, 2, ["RUNNING"], 1.4),
    ("GB", "Boxing Day Sales", 12, 26, 3, None, 1.8),
    ("JP", "Tokyo City Run Expo", 10, 24, 1, ["RUNNING"], 2.2),
    ("JP", "Golden Week", 4, 29, 7, None, 1.3),
    ("BR", "Carnaval", 2, 14, 5, ["LIFESTYLE", "APPAREL"], 1.3),
    ("BR", "São Paulo Night Run", 12, 31, 1, ["RUNNING"], 1.5),
    ("AU", "Harbour City Fun Run", 8, 9, 2, ["RUNNING"], 1.4),
    ("AU", "Boxing Day Sales", 12, 26, 3, None, 1.7),
    ("DE", "Spree River Marathon", 9, 21, 2, ["RUNNING"], 1.5),
    ("FR", "Les Soldes d'hiver", 1, 8, 14, None, 1.5),
    ("CA", "Boxing Week", 12, 26, 7, None, 1.6),
    ("CN", "Singles Day", 11, 11, 2, None, 2.4),
    ("KR", "Chuseok", 10, 5, 3, None, 1.2),
]

MONTH_FACTOR_N = [0.85, 0.80, 0.95, 0.97, 1.00, 1.00, 0.95, 1.12, 1.02, 0.96, 1.18, 1.35]
WEEKDAY_FACTOR = [0.82, 0.80, 0.85, 0.92, 1.12, 1.38, 1.18]   # Mon..Sun
FORMAT_FACTOR = {"FLAGSHIP": 2.4, "BRAND": 1.0, "COMMUNITY": 0.55, "FACTORY": 1.3, "ECOM": 4.2}

BANNED_REAL_BRANDS = ["nike", "jordan", "adidas", "puma", "reebok", "asics", "new balance", "under armour",
                      "foot locker", "jd sports", "dick's", "pegasus", "vaporfly", "air max", "dri-fit", "swoosh"]

In [ ]:
# Parameters: size profile + overrides from the widgets (blank widget = profile default)
PROFILES: dict[str, dict] = {
    # unit tests
    "tiny": dict(n_markets=5, stores_per_market=3, n_styles=12, history_days=75),
    # quick demo / laptop
    "small": dict(n_markets=5, stores_per_market=6, n_styles=24, history_days=120),
    # default for Databricks / Fabric demos
    "demo": dict(n_markets=6, stores_per_market=15, n_styles=60, history_days=365),
    # stress test (Spark sizing, partitioning)
    "large": dict(n_markets=10, stores_per_market=40, n_styles=150, history_days=730, demand_scale=1.5),
}


@dataclass(frozen=True)
class SynthParams:
    seed: int = 20261106
    as_of: str = "2026-10-31"          # "today" in the synthetic world; batch history ends the day before
    history_days: int = 120
    n_markets: int = 5                 # 5..10; US GB JP BR AU always first (they carry the scenario)
    stores_per_market: int = 6         # physical stores per market (+1 e-commerce node per market)
    n_styles: int = 24                 # 12..200
    demand_scale: float = 1.0          # multiplies unit demand
    dq_defect_rate: float = 0.0005     # share of fact rows that also get a defective copy
    brand: str = "Kinetiq"
    profile: str = "custom"

    @property
    def as_of_date(self) -> date:
        return date.fromisoformat(self.as_of)

    @property
    def history_start(self) -> date:
        return self.as_of_date - timedelta(days=self.history_days)

    @property
    def hero_launch(self) -> date:
        return self.as_of_date - timedelta(days=55)

    def validate(self) -> "SynthParams":
        if not 5 <= self.n_markets <= 10:
            raise ValueError("n_markets must be 5..10")
        if not 3 <= self.stores_per_market <= 60:
            raise ValueError("stores_per_market must be 3..60")
        if not 12 <= self.n_styles <= 200:
            raise ValueError("n_styles must be 12..200")
        if not 75 <= self.history_days <= 1095:
            raise ValueError("history_days must be 75..1095 (hero launch is as_of - 55 days)")
        if self.demand_scale <= 0:
            raise ValueError("demand_scale must be > 0")
        if not 0 <= self.dq_defect_rate <= 0.05:
            raise ValueError("dq_defect_rate must be 0..0.05")
        date.fromisoformat(self.as_of)
        return self

    def to_dict(self) -> dict:
        return asdict(self)

    def to_json(self) -> str:
        return json.dumps(self.to_dict(), sort_keys=True)

    def fingerprint(self) -> str:
        return hashlib.sha256(self.to_json().encode()).hexdigest()[:16]


_TYPES = {f.name: f.type for f in fields(SynthParams)}


def make_params(profile: str | None = "small", **overrides) -> SynthParams:
    """Profile defaults plus overrides. Empty-string / None overrides are ignored, so notebook widgets can be
    passed straight through."""
    values: dict = {}
    if profile:
        if profile not in PROFILES:
            raise ValueError(f"unknown profile {profile!r}; choose one of {sorted(PROFILES)}")
        values.update(PROFILES[profile])
        values["profile"] = profile
    for k, v in overrides.items():
        if v is None or (isinstance(v, str) and v.strip() == ""):
            continue
        if k not in _TYPES:
            raise ValueError(f"unknown parameter {k!r}")
        t = _TYPES[k]
        values[k] = int(v) if t in ("int", int) else float(v) if t in ("float", float) else str(v)
    return SynthParams(**values).validate()


params = make_params(dbutils.widgets.get("profile"),
                     **{k: dbutils.widgets.get(k) for k in ["seed", "as_of", "n_markets", "stores_per_market", "n_styles"]})

MODE = dbutils.widgets.get("mode")
ORDERS_PER_KEY_PARTNER = int(dbutils.widgets.get("orders_per_key_partner"))
ORDERS_PER_PARTNER = int(dbutils.widgets.get("orders_per_partner"))
PO_STYLE_SHARE = float(dbutils.widgets.get("po_style_share"))
MOVEMENT_DAYS = int(dbutils.widgets.get("movement_days"))
MOVEMENTS_PER_DAY = int(dbutils.widgets.get("movements_per_day"))
TABLE_PARAMS = dict(orders_per_key_partner=ORDERS_PER_KEY_PARTNER, orders_per_partner=ORDERS_PER_PARTNER,
                    po_style_share=PO_STYLE_SHARE, movement_days=MOVEMENT_DAYS, movements_per_day=MOVEMENTS_PER_DAY)
TABLE_DEFAULTS = dict(orders_per_key_partner=3, orders_per_partner=2, po_style_share=0.35, movement_days=14,
                      movements_per_day=60)
assert ORDERS_PER_KEY_PARTNER >= 1 and ORDERS_PER_PARTNER >= 1 and 0 <= PO_STYLE_SHARE <= 1
assert 1 <= MOVEMENT_DAYS <= 90 and 0 <= MOVEMENTS_PER_DAY <= 5000
print("mode:", MODE, "| table params:", TABLE_PARAMS)
print(params.to_json())
print("fingerprint:", params.fingerprint())

In [ ]:
# World: master data and the demo scenario (deterministic from the parameters)
SYNTHETIC_LABEL = "SYNTHETIC"
HERO_STYLE = "HM9443"
HERO_COLORS = ("001", "100")
HERO_SURGE_SIZES = ("9", "9.5", "10", "10.5", "11")
STORE_LANE_MAX_KM = 1500        # same-market store-to-store truck lanes only between stores this close
SCENARIO_MARKETS = ("US", "GB", "JP", "BR", "AU")


def rng_for(seed: int, *key) -> np.random.Generator:
    """Independent, order-free RNG stream per key (stable across processes and platforms)."""
    h = zlib.crc32("|".join(map(str, key)).encode())
    return np.random.default_rng([seed, h])


def gtin13(body12: str) -> str:
    digits = [int(c) for c in body12]
    s = sum(d * (3 if i % 2 else 1) for i, d in enumerate(digits))
    return body12 + str((10 - s % 10) % 10)


def round_price(usd: float, fx: float, premium: float, rule: str) -> float:
    v = usd * fx * premium
    if rule == "x.99":
        return float(math.ceil(v) - 0.01)
    if rule == "x9":
        return float(math.ceil(v / 10) * 10 - 1)
    step = int(rule)
    return float(math.ceil(v / step) * step)


@dataclass
class World:
    params: SynthParams
    markets: pd.DataFrame
    locations: pd.DataFrame
    styles: pd.DataFrame
    skus: pd.DataFrame
    assortment: pd.DataFrame          # location_id x style_color carried
    prices: pd.DataFrame
    partners: pd.DataFrame
    lanes: pd.DataFrame
    people: pd.DataFrame
    owners: pd.DataFrame
    vendors: pd.DataFrame
    events: pd.DataFrame
    store_managers: pd.DataFrame = field(default_factory=pd.DataFrame)   # location_id -> person_id (STORE only)
    scenario: dict = field(default_factory=dict)

    @property
    def market_ids(self) -> list[str]:
        return self.markets.market_id.tolist()

    def selling_locations(self, market_id: str | None = None) -> pd.DataFrame:
        loc = self.locations[self.locations.location_type.isin(["STORE", "ECOM"])]
        return loc if market_id is None else loc[loc.market_id == market_id]


# ---------------------------------------------------------------- markets & locations
def _markets(p: SynthParams) -> pd.DataFrame:
    rows = MARKETS[: p.n_markets]
    return pd.DataFrame(rows, columns=["market_id", "market_name", "region", "currency", "fx_usd_to_local",
                                       "price_premium", "hemisphere", "timezone", "price_rounding"])


def _formats(n: int) -> list[str]:
    n_flag = 1 if n >= 5 else 0
    n_fac = max(1, round(n * 0.15))
    n_com = max(1, round(n * 0.15)) if n >= 4 else 0
    n_brand = n - n_flag - n_fac - n_com
    return ["FLAGSHIP"] * n_flag + ["BRAND"] * n_brand + ["COMMUNITY"] * n_com + ["FACTORY"] * n_fac


def _dist(a, b) -> float:
    return math.hypot(a[0] - b[0], (a[1] - b[1]) * math.cos(math.radians(a[0])))


def _locations(p: SynthParams, markets: pd.DataFrame) -> pd.DataFrame:
    rows = []
    brand = p.brand
    dcs = [d for d in DCS if d[1] in set(markets.market_id) and (d[7] == 1 or p.stores_per_market >= 10)]
    for d in dcs:
        rows.append(dict(location_id=d[0], location_name=f"{brand} Distribution Center {d[2]}", location_type="DC",
                         format="DC", market_id=d[1], city=d[2], state_province=d[3], latitude=d[4], longitude=d[5],
                         timezone=d[6], serving_dc_id=None, opened_date=date(2008 + len(rows) % 9, 3, 1),
                         size_factor=0.0))
    for m in markets.itertuples():
        r = rng_for(p.seed, "loc", m.market_id)
        cities = CITIES[m.market_id]
        outlets = OUTLET_TOWNS[m.market_id]
        slots = [(c, d) for c in cities for d in c[5]]           # (city, district)
        # weight: first districts of each city first, so stores spread across cities before doubling up
        slots.sort(key=lambda s: s[0][5].index(s[1]))
        used = 0
        fac_i = 0
        mdc = [d for d in dcs if d[1] == m.market_id]
        for i, fmt in enumerate(_formats(p.stores_per_market)):
            store_no = 1001 + i
            lid = f"ST-{m.market_id}-{store_no}"
            if fmt == "FACTORY":
                town = outlets[fac_i % len(outlets)]
                fac_i += 1
                suffix = "" if fac_i <= len(outlets) else f" {fac_i // len(outlets) + 1}"
                city, state, lat, lon, tz = town
                name = f"{brand} Factory Store {city}{suffix}"
            else:
                (c, district) = slots[used % len(slots)]
                lap = used // len(slots)
                used += 1
                city, state, lat, lon, tz = c[0], c[1], c[2], c[3], c[4]
                lat += float(r.normal(0, 0.02))
                lon += float(r.normal(0, 0.02))
                extra = f" {lap + 1}" if lap else ""
                if fmt == "FLAGSHIP":
                    name = f"{brand} Flagship {city} {district}"
                elif fmt == "COMMUNITY":
                    name = f"{brand} Community Store {city} {district}{extra}"
                else:
                    name = f"{brand} {city} {district}{extra}"
            near = min(mdc, key=lambda d: _dist((lat, lon), (d[4], d[5])))
            rows.append(dict(location_id=lid, location_name=name, location_type="STORE", format=fmt,
                             market_id=m.market_id, city=city, state_province=state, latitude=round(lat, 5),
                             longitude=round(lon, 5), timezone=tz, serving_dc_id=near[0],
                             opened_date=date(int(r.integers(2004, 2025)), int(r.integers(1, 13)), 1),
                             size_factor=float(np.clip(r.lognormal(0, 0.28), 0.55, 1.9))))
        primary = mdc[0]
        rows.append(dict(location_id=f"EC-{m.market_id}", location_name=f"{brand}.com {m.market_name}",
                         location_type="ECOM", format="ECOM", market_id=m.market_id, city=primary[2],
                         state_province=primary[3], latitude=primary[4], longitude=primary[5], timezone=m.timezone,
                         serving_dc_id=primary[0], opened_date=date(2012, 6, 1), size_factor=1.0))
    df = pd.DataFrame(rows)
    df["record_label"] = SYNTHETIC_LABEL
    return df


# ---------------------------------------------------------------- products
def _styles(p: SynthParams) -> pd.DataFrame:
    r = rng_for(p.seed, "styles")
    as_of = p.as_of_date
    rows = [dict(style_id=HERO_STYLE, style_name="Men's AeroGlide 5", model="AeroGlide", category="RUNNING",
                 division="FOOTWEAR", gender="M", size_run="MENS_FW", msrp_usd=140.0,
                 launch_date=p.hero_launch, popularity=2.4, n_colors=2)]
    used_ids = {HERO_STYLE}
    used_names = {"Men's AeroGlide 5"}
    fam_w = np.array([f[6] for f in FAMILIES])
    fam_w = fam_w / fam_w.sum()
    while len(rows) < p.n_styles:
        fi = int(r.choice(len(FAMILIES), p=fam_w))
        cat, div, gender, models, tiers, run, pop = FAMILIES[fi]
        mi = int(r.integers(len(models)))
        model = models[mi]
        version = int(r.integers(2, 13)) if div == "FOOTWEAR" and not model[-1].isdigit() and "FG" not in model else None
        g = "Men's" if gender == "M" else "Women's"
        name = f"{g} {model}" + (f" {version}" if version else "")
        if name in used_names:
            name = f"{name} {['SE', 'Premium', 'GTX', 'Next Nature', 'Wide'][len(used_names) % 5]}"
            if name in used_names:
                continue
        while True:
            sid = f"{STYLE_PREFIXES[int(r.integers(len(STYLE_PREFIXES)))]}{int(r.integers(1000, 10000))}"
            if sid not in used_ids:
                break
        used_ids.add(sid)
        used_names.add(name)
        tier = tiers[min(mi, len(tiers) - 1)] if len(tiers) > mi else tiers[-1]
        # launch: mostly established product, ~20% launched inside the last 6 months
        age = int(r.integers(20, 180)) if r.random() < 0.2 else int(r.integers(180, 760))
        rows.append(dict(style_id=sid, style_name=name, model=model, category=cat, division=div, gender=gender,
                         size_run=run, msrp_usd=float(tier), launch_date=as_of - timedelta(days=age),
                         popularity=float(pop * r.lognormal(0, 0.45)),
                         n_colors=int(r.integers(3, 5)) if div == "APPAREL" else int(r.integers(2, 5))))
    df = pd.DataFrame(rows)
    age = (pd.Timestamp(as_of) - pd.to_datetime(df.launch_date)).dt.days
    df["lifecycle"] = np.where(age < 60, "LAUNCH", np.where(age > 540, "MARKDOWN", "CORE"))
    df["unit_cost_usd"] = (df.msrp_usd * df.division.map(UNIT_COST_RATIO)).round(2)
    df["record_label"] = SYNTHETIC_LABEL
    return df


def _size_weights(run: str) -> dict[str, float]:
    sizes = SIZE_RUNS[run]
    if run == "APPAREL":
        w = np.array([APPAREL_CURVE[s] for s in sizes])
    else:
        mu, sd = SIZE_CURVE_CENTER[run]
        w = np.array([math.exp(-0.5 * ((float(s) - mu) / sd) ** 2) for s in sizes])
    w = w / w.sum()
    return dict(zip(sizes, w))


def _skus(p: SynthParams, styles: pd.DataFrame) -> pd.DataFrame:
    rows = []
    seq = 0
    for s in styles.itertuples():
        r = rng_for(p.seed, "colors", s.style_id)
        if s.style_id == HERO_STYLE:
            colors = [c for c in COLORWAYS if c[0] in HERO_COLORS]
            shares = [0.62, 0.38]
        else:
            idx = sorted(r.choice(len(COLORWAYS), s.n_colors, replace=False))
            colors = [COLORWAYS[i] for i in idx]
            shares = list(r.dirichlet(np.full(len(colors), 2.5)))
        sw = _size_weights(s.size_run)
        for (code, cname), cshare in zip(colors, shares):
            for size, w in sw.items():
                seq += 1
                rows.append(dict(sku_id=f"{s.style_id}-{code}-{size}", style_id=s.style_id, color_code=code,
                                 color_name=cname, size=size, size_run=s.size_run,
                                 gtin=gtin13(f"029{seq:09d}"), color_share=float(cshare), size_weight=float(w)))
    df = pd.DataFrame(rows)
    df["record_label"] = SYNTHETIC_LABEL
    return df


def _assortment(p: SynthParams, locations: pd.DataFrame, styles: pd.DataFrame, skus: pd.DataFrame) -> pd.DataFrame:
    sc = skus[["style_id", "color_code"]].drop_duplicates()
    sc = sc.merge(styles[["style_id", "lifecycle", "launch_date"]], on="style_id")
    rows = []
    for loc in locations[locations.location_type.isin(["STORE", "ECOM"])].itertuples():
        r = rng_for(p.seed, "assort", loc.location_id)
        u = r.random(len(sc))
        if loc.format in ("FLAGSHIP", "ECOM"):
            keep = np.ones(len(sc), bool)
        elif loc.format == "FACTORY":
            keep = (sc.lifecycle.to_numpy() != "LAUNCH") & (u < 0.75)
        else:
            rate = 0.8 if loc.format == "BRAND" else 0.55
            keep = (u < rate) & (sc.lifecycle.to_numpy() != "MARKDOWN") | (sc.style_id.to_numpy() == HERO_STYLE)
        part = sc.loc[keep, ["style_id", "color_code"]].copy()
        part["location_id"] = loc.location_id
        rows.append(part)
    return pd.concat(rows, ignore_index=True)[["location_id", "style_id", "color_code"]]


def _prices(styles: pd.DataFrame, markets: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for s in styles.itertuples():
        for m in markets.itertuples():
            rows.append(dict(style_id=s.style_id, market_id=m.market_id, currency=m.currency,
                             full_price_local=round_price(s.msrp_usd, m.fx_usd_to_local, m.price_premium, m.price_rounding),
                             markdown_pct=0.30 if s.lifecycle == "MARKDOWN" else 0.0,
                             effective_from=min(s.launch_date, date(2025, 1, 1)) if s.lifecycle != "LAUNCH" else s.launch_date))
    df = pd.DataFrame(rows)
    df["record_label"] = SYNTHETIC_LABEL
    return df


# ---------------------------------------------------------------- partners, lanes, people
def _partners(markets: pd.DataFrame, locations: pd.DataFrame) -> pd.DataFrame:
    mk = set(markets.market_id)
    rows = []
    for i, (name, m, tier) in enumerate(WHOLESALE_PARTNERS):
        if m not in mk:
            continue
        dc = locations[(locations.market_id == m) & (locations.location_type == "DC")].location_id.iloc[0]
        rows.append(dict(partner_id=f"WA-{1001 + i}", partner_name=name, market_id=m, tier=tier, ship_from_dc_id=dc))
    df = pd.DataFrame(rows)
    df["record_label"] = SYNTHETIC_LABEL
    return df


def _lanes(p: SynthParams, locations: pd.DataFrame) -> pd.DataFrame:
    rows = []
    dcs = locations[locations.location_type == "DC"]
    for a in dcs.itertuples():
        for b in dcs.itertuples():
            if a.location_id == b.location_id:
                continue
            same = a.market_id == b.market_id
            km = _dist((a.latitude, a.longitude), (b.latitude, b.longitude)) * 111
            rows.append(dict(from_location_id=a.location_id, to_location_id=b.location_id,
                             mode="TRUCK" if same else ("AIR" if km > 6000 else "OCEAN_AIR"),
                             lead_days=int(2 + km // 900) if same else int(min(14, 5 + km // 2500)),
                             cost_per_unit_usd=round(0.35 + km * (0.00012 if same else 0.00055), 2)))
    for s in locations[locations.location_type.isin(["STORE", "ECOM"])].itertuples():
        dc = locations.set_index("location_id").loc[s.serving_dc_id]
        km = _dist((s.latitude, s.longitude), (dc.latitude, dc.longitude)) * 111
        for a, b in ((s.serving_dc_id, s.location_id), (s.location_id, s.serving_dc_id)):
            rows.append(dict(from_location_id=a, to_location_id=b, mode="TRUCK", lead_days=int(1 + km // 700),
                             cost_per_unit_usd=round(0.25 + km * 0.0002, 2)))
    # Store-to-store rebalancing lanes: same market, within a day or two by truck (UC1 store extension)
    stores = locations[locations.location_type == "STORE"]
    for a in stores.itertuples():
        for b in stores.itertuples():
            if a.location_id == b.location_id or a.market_id != b.market_id:
                continue
            km = _dist((a.latitude, a.longitude), (b.latitude, b.longitude)) * 111
            if km > STORE_LANE_MAX_KM:
                continue
            rows.append(dict(from_location_id=a.location_id, to_location_id=b.location_id, mode="TRUCK",
                             lead_days=int(1 + km // 800), cost_per_unit_usd=round(0.6 + km * 0.00025, 2)))
    df = pd.DataFrame(rows)
    df["record_label"] = SYNTHETIC_LABEL
    return df


def _people_owners(markets: pd.DataFrame, styles: pd.DataFrame):
    people = pd.DataFrame(PEOPLE, columns=["person_id", "display_name", "title", "home_market_id"])
    people["email"] = people.person_id + "@kinetiq.example"
    people["record_label"] = SYNTHETIC_LABEL
    rows = []
    cats = sorted(styles.category.unique())
    for m in markets.market_id:
        for c in cats:
            if (m, c) == ("BR", "TRAIL"):
                pass                                          # DQ-12: no accountable owner
            else:
                rows.append((c, m, "ACCOUNTABLE", MARKET_DIRECTOR[m], 5000))
            if (m, c) == ("AU", "LIFESTYLE"):
                rows.append((c, m, "ACCOUNTABLE", "gkim", 2000))  # DQ-12: two accountable owners
            rows.append((c, m, "PLANNER", MARKET_PLANNER[m], 500))
            rows.append((c, m, "WHOLESALE_LEAD", WHOLESALE_LEAD[m], 0))
            rows.append((c, m, "FINANCE", "ohaddad", 0))
    owners = pd.DataFrame(rows, columns=["category", "market_id", "role", "person_id", "approval_limit_units"])
    owners["record_label"] = SYNTHETIC_LABEL
    return people, owners


def _store_managers(p: SynthParams, locations: pd.DataFrame, people: pd.DataFrame):
    """One invented manager per store: extra ref.person rows plus the ref.store_manager mapping."""
    r = rng_for(p.seed, "store-manager")
    rows, links = [], []
    for s in locations[locations.location_type == "STORE"].itertuples():
        pid = "sm-" + s.location_id[3:].lower()                     # ST-US-1001 -> sm-us-1001
        name = f"{r.choice(STORE_MANAGER_FIRST)} {r.choice(STORE_MANAGER_LAST)}"
        rows.append(dict(person_id=pid, display_name=name, title=f"Store Manager, {s.location_name}"[:80],
                         home_market_id=s.market_id, email=f"{pid}@kinetiq.example", record_label=SYNTHETIC_LABEL))
        links.append(dict(location_id=s.location_id, person_id=pid, record_label=SYNTHETIC_LABEL))
    return pd.concat([people, pd.DataFrame(rows)], ignore_index=True), pd.DataFrame(links)


def _events(p: SynthParams, markets: pd.DataFrame) -> pd.DataFrame:
    rows = []
    mk = set(markets.market_id)
    for y in range(p.history_start.year, p.as_of_date.year + 1):
        for (m, name, mo, d, dur, cats, up) in REGIONAL_EVENTS:
            if m not in mk:
                continue
            start = date(y, mo, d)
            end = start + timedelta(days=dur - 1)
            if end < p.history_start or start > p.as_of_date + timedelta(days=30):
                continue
            rows.append(dict(event_id=f"EV-{m}-{start:%Y%m%d}", market_id=m, event_name=name, start_date=start,
                             end_date=end, categories=",".join(cats) if cats else "ALL", demand_uplift=up))
    df = pd.DataFrame(rows)
    df["record_label"] = SYNTHETIC_LABEL
    return df


def build_world(p: SynthParams) -> World:
    p.validate()
    markets = _markets(p)
    locations = _locations(p, markets)
    styles = _styles(p)
    skus = _skus(p, styles)
    people, owners = _people_owners(markets, styles)
    people, store_managers = _store_managers(p, locations, people)
    vendors = pd.DataFrame(VENDORS, columns=["vendor_id", "vendor_name", "country", "lead_days"])
    vendors["record_label"] = SYNTHETIC_LABEL
    w = World(params=p, markets=markets, locations=locations, styles=styles, skus=skus,
              assortment=_assortment(p, locations, styles, skus), prices=_prices(styles, markets),
              partners=_partners(markets, locations), lanes=_lanes(p, locations), people=people, owners=owners,
              vendors=vendors, events=_events(p, markets), store_managers=store_managers)
    w.scenario = dict(
        hero_style=HERO_STYLE, hero_colors=list(HERO_COLORS), hero_launch=p.hero_launch.isoformat(),
        surge_market="US", surge_sizes=list(HERO_SURGE_SIZES), surge_from_day=18, surge_sales=1.8, surge_intent=1.6,
        slow_market="GB", slow_sales=0.3, slow_intent=0.6,
        spike_market="JP", spike_event="Tokyo City Run Expo", spike_intent=3.2,
        stale_market="BR", stale_days=2,
        restated_date=(p.as_of_date - timedelta(days=3)).isoformat(),
        reserve_partner="WA-1003", cancel_partner="WA-1004", slipped_po="4577091802", po_slip_days=9,
        negative_stock_location="ST-AU-1002", overcommit_dc="DC-AU-TRU",
        # Store extension: the slowest hero stores in the surge market were over-allocated at launch and hold 9-12 units
        # of each surge size (DC spare first, the rest is part of the launch allocation)
        donor_store_count=3, donor_store_units=(9, 12),
    )
    return w


world = build_world(params)
print(f"markets={world.market_ids}  locations={len(world.locations)}  styles={len(world.styles)}  "
      f"skus={len(world.skus)}  assortment={len(world.assortment)}")

In [ ]:
# Expected-demand model (seasonality, weekday, lifecycle, events, scenario multipliers)
BASE_UNITS_PER_STORE_DAY = 70.0     # a typical BRAND store sells ~70 units/day across footwear + apparel


class MarketDemand:
    """Pre-computed (location x sku) base rates for one market. `rates(days)` -> array [day, loc, sku]."""

    def __init__(self, w: World, market_id: str):
        self.w = w
        self.market_id = market_id
        p = w.params
        m = w.markets.set_index("market_id").loc[market_id]
        self.hemisphere = m.hemisphere
        self.locs = w.selling_locations(market_id).reset_index(drop=True)
        skus = w.skus.merge(w.styles[["style_id", "category", "division", "launch_date", "lifecycle", "popularity"]],
                            on="style_id")
        self.skus = skus.reset_index(drop=True)
        n_l, n_s = len(self.locs), len(self.skus)

        carried = np.zeros((n_l, n_s), bool)
        key = self.skus.style_id + "|" + self.skus.color_code
        for i, loc in enumerate(self.locs.location_id):
            a = w.assortment[w.assortment.location_id == loc]
            carried[i] = key.isin(set(a.style_id + "|" + a.color_code)).to_numpy()
        self.carried = carried

        style_share = self.skus.popularity.to_numpy() / w.styles.popularity.sum()
        sku_share = style_share * self.skus.color_share.to_numpy() * self.skus.size_weight.to_numpy()
        fmt = self.locs.format.map(FORMAT_FACTOR).to_numpy() * self.locs.size_factor.to_numpy()
        units = BASE_UNITS_PER_STORE_DAY * p.demand_scale * fmt
        # renormalise per location so each location's carried assortment sells its target volume
        raw = carried * sku_share[None, :]
        denom = raw.sum(axis=1, keepdims=True)
        denom[denom == 0] = 1
        self.base = raw / denom * units[:, None]                     # [loc, sku]

        self.launch = pd.to_datetime(self.skus.launch_date).to_numpy().astype("datetime64[D]")
        self.category = self.skus.category.to_numpy()
        self.lifecycle = self.skus.lifecycle.to_numpy()
        self.is_hero = (self.skus.style_id == HERO_STYLE).to_numpy()
        self.hero_surge = self.is_hero & self.skus["size"].isin(HERO_SURGE_SIZES).to_numpy()
        ev = w.events[w.events.market_id == market_id]
        self.events = [(np.datetime64(e.start_date), np.datetime64(e.end_date),
                        None if e.categories == "ALL" else set(e.categories.split(",")), e.demand_uplift)
                       for e in ev.itertuples()]

    def day_factors(self, d: date, scenario: bool = True) -> np.ndarray:
        """Multiplicative factor per sku for calendar day d (season, weekday, lifecycle, events, scenario)."""
        w = self.w
        mi = d.month - 1 if self.hemisphere == "N" else (d.month + 5) % 12
        f = np.full(len(self.skus), MONTH_FACTOR_N[mi] * WEEKDAY_FACTOR[d.weekday()])
        d64 = np.datetime64(d)
        age = (d64 - self.launch).astype(int)
        life = np.where(age < 0, 0.0, np.where(age < 30, 1.0 + 0.8 * (1 - age / 30), 1.0))
        life = np.where(age > 540, 0.6, life)
        life = np.where(self.lifecycle == "MARKDOWN", life * 1.25, life)
        f = f * life
        for start, end, cats, up in self.events:
            if start <= d64 <= end:
                f = f * (np.where(np.isin(self.category, list(cats)), up, 1.0) if cats else up)
        if not scenario:
            return f
        sc = w.scenario
        hero_day = (d64 - np.datetime64(sc["hero_launch"])).astype(int)
        if self.market_id == sc["surge_market"] and hero_day >= sc["surge_from_day"]:
            f = np.where(self.hero_surge, f * sc["surge_sales"], f)
        if self.market_id == sc["slow_market"]:
            f = np.where(self.is_hero, f * sc["slow_sales"], f)
        return f

    def rates(self, days: list[date], scenario: bool = True) -> np.ndarray:
        return np.stack([self.base * self.day_factors(d, scenario)[None, :] for d in days])


def month_starts(start: date, end: date) -> list[date]:
    """First day of every calendar month overlapping [start, end]."""
    out, cur = [], date(start.year, start.month, 1)
    while cur <= end:
        out.append(cur)
        cur = date(cur.year + (cur.month == 12), cur.month % 12 + 1, 1)
    return out


def days_in_chunk(w: World, month_start: date) -> list[date]:
    p = w.params
    nxt = date(month_start.year + (month_start.month == 12), month_start.month % 12 + 1, 1)
    first = max(month_start, p.history_start)
    last = min(nxt - timedelta(days=1), p.as_of_date - timedelta(days=1))
    return [first + timedelta(days=i) for i in range((last - first).days + 1)]


_DEMAND: dict = {}


def demand_for(market_id: str) -> MarketDemand:
    if market_id not in _DEMAND:
        _DEMAND[market_id] = MarketDemand(world, market_id)
    return _DEMAND[market_id]

## ERP/WMS tables

Every table gets a primary key (required for Fabric mirroring) and `record_label = 'SYNTHETIC'`.

In [ ]:
LAUNCH_MARGIN_FLOOR = 0.55
LAUNCH_PROTECTION_DAYS = 60
APPAREL_VENDORS = ("KH-PPA", "BD-DHA")

# Primary keys for every Azure SQL table (mirroring requires a PK; tests assert uniqueness).
PRIMARY_KEYS: dict[str, list[str]] = {
    "ref.market": ["market_id"],
    "ref.person": ["person_id"],
    "ref.decision_owner": ["owner_row_id"],
    "ref.synthetic_run": ["run_id"],
    "erp.location": ["location_id"],
    "erp.product_style": ["style_id"],
    "erp.product_sku": ["sku_id"],
    "erp.price": ["style_id", "market_id"],
    "erp.vendor": ["vendor_id"],
    "erp.wholesale_partner": ["partner_id"],
    "erp.wholesale_order": ["order_id"],
    "erp.wholesale_order_line": ["order_id", "line_no"],
    "erp.purchase_order": ["po_id"],
    "erp.purchase_order_line": ["po_id", "line_no"],
    "erp.margin_guardrail": ["market_id", "category", "lifecycle"],
    "erp.regional_event": ["event_id"],
    "wms.inventory_position": ["location_id", "sku_id"],
    "wms.transfer_lane": ["from_location_id", "to_location_id"],
    "wms.inventory_movement": ["movement_id"],
    "ref.store_manager": ["location_id"],
}


def _label(df: pd.DataFrame) -> pd.DataFrame:
    df["record_label"] = SYNTHETIC_LABEL
    return df

TABLES: dict = {}
DEMAND = {m: demand_for(m) for m in world.market_ids}

### Master data (from the world model)

In [ ]:
TABLES["ref.market"] = _label(world.markets.copy())
print(len(TABLES["ref.market"]), "markets")

In [ ]:
# people incl. one invented manager per store
TABLES["ref.person"] = world.people
print(len(TABLES["ref.person"]), "people")

In [ ]:
# owners per category x market (DQ-12: BR/TRAIL has no accountable owner, AU/LIFESTYLE has two)
TABLES["ref.decision_owner"] = world.owners.reset_index(names="owner_row_id")
print(len(TABLES["ref.decision_owner"]), "owner rows")

In [ ]:
TABLES["ref.store_manager"] = world.store_managers
print(len(TABLES["ref.store_manager"]), "store managers")

In [ ]:
# DCs, stores (flagship / brand / community / factory) and one e-commerce node per market
TABLES["erp.location"] = world.locations.drop(columns=["size_factor"])
display(TABLES["erp.location"].groupby(["market_id", "location_type"]).size().unstack(fill_value=0))

In [ ]:
TABLES["erp.product_style"] = world.styles.drop(columns=["popularity", "n_colors"])
print(len(TABLES["erp.product_style"]), "styles")

In [ ]:
TABLES["erp.product_sku"] = world.skus.drop(columns=["color_share", "size_weight"])
print(len(TABLES["erp.product_sku"]), "SKUs")

In [ ]:
TABLES["erp.price"] = world.prices
print(len(TABLES["erp.price"]), "prices")

In [ ]:
TABLES["erp.vendor"] = world.vendors
print(len(TABLES["erp.vendor"]), "vendors")

In [ ]:
TABLES["erp.wholesale_partner"] = world.partners
print(len(TABLES["erp.wholesale_partner"]), "wholesale partners")

In [ ]:
TABLES["erp.regional_event"] = world.events
print(len(TABLES["erp.regional_event"]), "regional events")

In [ ]:
# DC<->DC, store<->DC and same-market store<->store truck lanes
TABLES["wms.transfer_lane"] = world.lanes
print(len(TABLES["wms.transfer_lane"]), "lanes")

### `wms.inventory_position`

Store and DC stock from the recent expected demand x weeks of cover. Scenario: US hero surge sizes are short, GB hero is overstocked; DQ-04 negative on-hand, DQ-09 over-commit.

In [ ]:
def _recent_rate(w: World, md: MarketDemand, days: int = 14) -> np.ndarray:
    """Average expected daily units [loc, sku] over the last `days` before as_of (scenario included)."""
    as_of = w.params.as_of_date
    ds = [as_of - timedelta(days=i) for i in range(1, days + 1)]
    return md.rates(ds).mean(axis=0)


# ------------------------------------------------------------------ WMS inventory
def inventory_position(w: World, demand: dict[str, MarketDemand]) -> pd.DataFrame:
    p, sc = w.params, w.scenario
    snap = datetime.combine(p.as_of_date, dtime(5, 0))
    frames = []
    for m in w.market_ids:
        md = demand[m]
        r = rng_for(p.seed, "inv", m)
        rate = _recent_rate(w, md)
        woc = np.clip(r.lognormal(np.log(4.0), 0.35, rate.shape), 1.2, 12)
        hero = md.is_hero[None, :]
        if m == sc["surge_market"]:
            woc = np.where(md.hero_surge[None, :], r.uniform(0.4, 1.4, rate.shape), woc)
        if m == sc["slow_market"]:
            woc = np.where(hero, r.uniform(8.5, 14, rate.shape), woc)
        on_hand = np.where(md.carried, np.round(rate * woc * 7), 0).astype(int)
        li, si = np.nonzero(md.carried)
        frames.append(pd.DataFrame({
            "location_id": md.locs.location_id.to_numpy()[li], "sku_id": md.skus.sku_id.to_numpy()[si],
            "on_hand_units": on_hand[li, si], "allocated_units": r.binomial(on_hand[li, si], 0.05),
            "reserved_units": 0, "in_transit_units": np.where(r.random(len(li)) < 0.15,
                                                               r.integers(1, 6, len(li)), 0),
            "_dc_rate": rate[li, si]}))
        # DC stock: market-level SKU demand x DC weeks of cover, split across the market's DCs
        dcs = w.locations[(w.locations.market_id == m) & (w.locations.location_type == "DC")].location_id.tolist()
        mkt_rate = rate.sum(axis=0)
        dc_woc = np.clip(r.lognormal(np.log(6.0), 0.3, len(mkt_rate)), 2, 16)
        if m == sc["surge_market"]:
            dc_woc = np.where(md.hero_surge, r.uniform(0.8, 1.6, len(mkt_rate)), dc_woc)
        if m == sc["slow_market"]:
            dc_woc = np.where(md.is_hero, r.uniform(10, 16, len(mkt_rate)), dc_woc)
        share = np.array([0.7] + [0.3 / (len(dcs) - 1)] * (len(dcs) - 1)) if len(dcs) > 1 else np.array([1.0])
        for dc, s in zip(dcs, share):
            oh = np.round(mkt_rate * dc_woc * 7 * s).astype(int)
            frames.append(pd.DataFrame({"location_id": dc, "sku_id": md.skus.sku_id.to_numpy(), "on_hand_units": oh,
                                        "allocated_units": r.binomial(oh, 0.18), "reserved_units": 0,
                                        "in_transit_units": 0, "_dc_rate": mkt_rate * s}))
    inv = pd.concat(frames, ignore_index=True)
    r = rng_for(p.seed, "inv-dq")
    # DQ-04 negative on-hand (POS sold before receipt was booked)
    neg = inv.index[inv.location_id == sc["negative_stock_location"]]
    pick = r.choice(neg, min(3, len(neg)), replace=False)
    inv.loc[pick, "on_hand_units"] = -r.integers(1, 4, len(pick))
    inv.loc[pick, "allocated_units"] = 0
    # DQ-09 over-commit: allocated + reserved > on_hand at one DC
    oc = inv.index[(inv.location_id == sc["overcommit_dc"]) & (inv.on_hand_units > 10)]
    pick = r.choice(oc, min(4, len(oc)), replace=False)
    inv.loc[pick, "allocated_units"] = (inv.loc[pick, "on_hand_units"] * 1.15).round().astype(int) + 2
    inv["snapshot_ts"] = snap
    inv["inventory_status"] = "AVAILABLE"
    return _label(inv.drop(columns="_dc_rate"))


TABLES["wms.inventory_position"] = inventory_position(world, DEMAND)
print(len(TABLES["wms.inventory_position"]), "inventory rows")

### `erp.wholesale_order` and `erp.wholesale_order_line`

One order document (header + lines). Scenario: partner WA-1003 reserves hero units (held at the ship-from DC), WA-1004 cancels; DQ-11 re-sent cancelled line.

In [ ]:
def wholesale_orders(w: World, inv: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    p, sc = w.params, w.scenario
    as_of = p.as_of_date
    hdr, lines = [], []
    order_no = 7100000
    top = w.styles.sort_values("popularity", ascending=False).style_id.tolist()
    for pt in w.partners.itertuples():
        r = rng_for(p.seed, "wo", pt.partner_id)
        n_orders = ORDERS_PER_KEY_PARTNER if pt.tier == "KEY" else ORDERS_PER_PARTNER
        for k in range(n_orders):
            order_no += 1
            oid = f"SO{order_no}"
            created = as_of - timedelta(days=int(r.integers(3, 40)))
            ship = as_of + timedelta(days=int(r.integers(5, 45)))
            status = "OPEN"
            styles = list(r.choice(top[: max(6, len(top) // 2)], int(r.integers(2, 5)), replace=False))
            if pt.partner_id == sc["reserve_partner"] and k == 0:
                styles = [HERO_STYLE] + [s for s in styles if s != HERO_STYLE][:1]
                status = "RESERVED"
            if pt.partner_id == sc["cancel_partner"] and k == 0:
                styles = [HERO_STYLE]
                status = "CANCELLED"
            hdr.append(dict(order_id=oid, partner_id=pt.partner_id, market_id=pt.market_id,
                            ship_from_location_id=pt.ship_from_dc_id, order_date=created, requested_ship_date=ship,
                            order_status=status, currency=w.markets.set_index("market_id").currency[pt.market_id]))
            ln = 0
            for st in styles:
                sk = w.skus[w.skus.style_id == st]
                sk = sk[sk.color_code == sk.color_code.iloc[0]] if st != HERO_STYLE else sk[sk.color_code == "001"]
                for s in sk.itertuples():
                    q = int(r.poisson(18 * s.size_weight * (3 if pt.tier == "KEY" else 1.5) * len(sk)))
                    if q == 0:
                        continue
                    ln += 1
                    lines.append(dict(order_id=oid, line_no=ln, sku_id=s.sku_id, ordered_units=q,
                                      line_status=status, source_message_id=f"EDI850-{oid}-{ln:03d}"))
    hdr = pd.DataFrame(hdr)
    lines = pd.DataFrame(lines)
    # DQ-11: one cancelled line re-sent by the EDI bridge as OPEN under a new line number
    canc = lines[lines.line_status == "CANCELLED"]
    if len(canc):
        dup = canc.iloc[[0]].copy()
        dup["line_no"] = lines[lines.order_id == dup.order_id.iloc[0]].line_no.max() + 1
        dup["line_status"] = "OPEN"
        dup["source_message_id"] = dup.source_message_id.iloc[0] + "-RESEND"
        lines = pd.concat([lines, dup], ignore_index=True)
    # Reserved hero units are held at the ship-from DC
    res = lines[lines.line_status == "RESERVED"].merge(hdr[["order_id", "ship_from_location_id"]], on="order_id")
    for x in res.itertuples():
        m = (inv.location_id == x.ship_from_location_id) & (inv.sku_id == x.sku_id)
        inv.loc[m, "reserved_units"] = inv.loc[m, "reserved_units"] + x.ordered_units
    return _label(hdr), _label(lines)


TABLES["erp.wholesale_order"], TABLES["erp.wholesale_order_line"] = wholesale_orders(world, TABLES["wms.inventory_position"])
print(len(TABLES["erp.wholesale_order"]), "orders", len(TABLES["erp.wholesale_order_line"]), "lines")

### Donor stores (updates `wms.inventory_position`)

Store-to-store extension: the slowest hero stores in the surge market were over-allocated the surge sizes at launch, so they can give stock to stores that are running out.

In [ ]:
def donor_stores(w: World, demand: dict[str, MarketDemand], inv: pd.DataFrame) -> list[str]:
    """Store extension of UC1: the slowest-selling hero stores in the surge market were over-allocated the surge sizes.

    Each donor holds a launch allocation of a few units per surge size, far more than its slow sales need. Units come
    from the market's DCs first (only stock that is not allocated or reserved, so no DC becomes over-committed);
    because the surge sizes have little DC spare, the remainder is booked as part of the launch allocation, which lifts
    market stock slightly (the market stays at T1 risk). Runs after the wholesale reservations; uses its own RNG stream
    so every other draw is unchanged.
    """
    p, sc = w.params, w.scenario
    m, n = sc["surge_market"], sc.get("donor_store_count", 0)
    if not n:
        return []
    md = demand[m]
    rd = rng_for(p.seed, "inv-donor")
    rate = _recent_rate(w, md)
    is_store = md.locs.location_type.eq("STORE").to_numpy()
    hero_rate = np.where(md.carried & md.hero_surge[None, :], rate, 0).sum(axis=1)
    cand = np.flatnonzero(is_store & (hero_rate > 0))
    donors = cand[np.argsort(hero_rate[cand], kind="stable")][:n]
    lo, hi = sc["donor_store_units"]
    dcs = w.locations[(w.locations.market_id == m) & (w.locations.location_type == "DC")].location_id
    for j in np.flatnonzero(md.hero_surge):
        sku = md.skus.sku_id.iloc[j]
        dc_rows = inv.index[inv.location_id.isin(dcs) & (inv.sku_id == sku)]
        spare = (inv.loc[dc_rows, "on_hand_units"] - inv.loc[dc_rows, "allocated_units"]
                 - inv.loc[dc_rows, "reserved_units"]).clip(lower=0)
        for li in donors:
            if not md.carried[li, j]:
                continue
            row = inv.index[(inv.location_id == md.locs.location_id.iloc[li]) & (inv.sku_id == sku)][0]
            want = int(rd.integers(lo, hi + 1)) - int(inv.at[row, "on_hand_units"])
            for d in spare.sort_values(ascending=False).index:
                take = int(min(max(want, 0), spare[d]))
                if take <= 0:
                    continue
                inv.at[d, "on_hand_units"] -= take
                inv.at[row, "on_hand_units"] += take
                spare[d] -= take
                want -= take
            if want > 0:
                inv.at[row, "on_hand_units"] += want
    return md.locs.location_id.iloc[donors].tolist()


DONORS = donor_stores(world, DEMAND, TABLES["wms.inventory_position"])
print("donor stores:", DONORS)

### `erp.purchase_order` and `erp.purchase_order_line`

Scenario: hero PO 4577091802 to the US DC has slipped 9 days.

In [ ]:
def purchase_orders(w: World) -> tuple[pd.DataFrame, pd.DataFrame]:
    p, sc = w.params, w.scenario
    as_of = p.as_of_date
    r = rng_for(p.seed, "po")
    hdr, lines = [], []
    po_no = 4577092000                   # sequence starts above the scenario's fixed hero PO id
    dcs = w.locations[(w.locations.location_type == "DC")]
    primary = dcs.drop_duplicates("market_id")
    for st in w.styles.itertuples():
        if st.lifecycle == "MARKDOWN":
            continue
        for dc in primary.itertuples():
            if r.random() > (1.0 if st.style_id == HERO_STYLE else PO_STYLE_SHARE):
                continue
            po_no += 1
            vendor = w.vendors.iloc[int(r.integers(len(w.vendors)))]
            if st.division == "APPAREL":
                vendor = w.vendors[w.vendors.vendor_id.isin(APPAREL_VENDORS)].iloc[int(r.integers(2))]
            else:
                fw = w.vendors[~w.vendors.vendor_id.isin(APPAREL_VENDORS)]
                vendor = fw.iloc[int(r.integers(len(fw)))]
            promise = as_of + timedelta(days=int(r.integers(7, 70)))
            pid = str(po_no)
            if st.style_id == HERO_STYLE and dc.market_id == sc["surge_market"]:
                pid = sc["slipped_po"]
                promise = as_of + timedelta(days=12)
            slip = sc["po_slip_days"] if pid == sc["slipped_po"] else (int(r.integers(0, 4)) if r.random() < 0.2 else 0)
            hdr.append(dict(po_id=pid, vendor_id=vendor.vendor_id, ship_to_location_id=dc.location_id,
                            style_id=st.style_id, po_date=promise - timedelta(days=int(vendor.lead_days) + 30),
                            original_promise_date=promise, current_promise_date=promise + timedelta(days=slip),
                            po_status="IN_TRANSIT" if (promise - as_of).days < 25 else "CONFIRMED",
                            incoterm="FOB"))
            sk = w.skus[w.skus.style_id == st.style_id]
            for i, s in enumerate(sk.itertuples(), 1):
                lines.append(dict(po_id=pid, line_no=i, sku_id=s.sku_id,
                                  ordered_units=int(max(6, round(900 * s.color_share * s.size_weight))),
                                  unit_cost_usd=float(w.styles.set_index("style_id").unit_cost_usd[st.style_id])))
    hdr = pd.DataFrame(hdr).drop_duplicates("po_id")
    lines = pd.DataFrame(lines)
    return _label(hdr), _label(lines[lines.po_id.isin(hdr.po_id)])


TABLES["erp.purchase_order"], TABLES["erp.purchase_order_line"] = purchase_orders(world)
print(len(TABLES["erp.purchase_order"]), "POs", len(TABLES["erp.purchase_order_line"]), "lines")

### `erp.margin_guardrail`

In [ ]:
def margin_guardrails(w: World) -> pd.DataFrame:
    rows = []
    for m in w.market_ids:
        for c in sorted(w.styles.category.unique()):
            rows.append(dict(market_id=m, category=c, lifecycle="LAUNCH", min_gross_margin_pct=LAUNCH_MARGIN_FLOOR,
                             protection_days=LAUNCH_PROTECTION_DAYS, max_transfer_cost_pct=0.06))
            rows.append(dict(market_id=m, category=c, lifecycle="CORE", min_gross_margin_pct=0.45,
                             protection_days=0, max_transfer_cost_pct=0.08))
            rows.append(dict(market_id=m, category=c, lifecycle="MARKDOWN", min_gross_margin_pct=0.25,
                             protection_days=0, max_transfer_cost_pct=0.10))
    return _label(pd.DataFrame(rows))


TABLES["erp.margin_guardrail"] = margin_guardrails(world)
print(len(TABLES["erp.margin_guardrail"]), "guardrails")

### `wms.inventory_movement`

Recent DC -> store replenishment (WMS transaction log).

In [ ]:
def inventory_movements(w: World, inv: pd.DataFrame, days: int = 14, per_day: int = 60) -> pd.DataFrame:
    """Recent DC -> store replenishment and vendor receipts (WMS transaction log)."""
    p = w.params
    r = rng_for(p.seed, "mv")
    stores = w.locations[w.locations.location_type == "STORE"].set_index("location_id")
    st_inv = inv[inv.location_id.isin(stores.index) & (inv.on_hand_units > 0)]
    rows = []
    mv = 90000000
    for d in range(days, 0, -1):
        day = p.as_of_date - timedelta(days=d)
        sample = st_inv.sample(n=min(len(st_inv), per_day), random_state=int(r.integers(1 << 31)))
        for x in sample.itertuples():
            mv += 1
            rows.append(dict(movement_id=f"MV{mv}", movement_ts=datetime.combine(day, dtime(int(r.integers(4, 9)))),
                             movement_type="REPLENISH", sku_id=x.sku_id,
                             from_location_id=stores.serving_dc_id[x.location_id], to_location_id=x.location_id,
                             units=int(r.integers(1, 7)), reference_id=None))
    return _label(pd.DataFrame(rows))


TABLES["wms.inventory_movement"] = inventory_movements(world, TABLES["wms.inventory_position"], MOVEMENT_DAYS, MOVEMENTS_PER_DAY)
print(len(TABLES["wms.inventory_movement"]), "movements")

### `ref.synthetic_run`

In [ ]:
pj = params.to_dict()
if TABLE_PARAMS != TABLE_DEFAULTS:
    pj["table_params"] = TABLE_PARAMS
TABLES["ref.synthetic_run"] = _label(pd.DataFrame([dict(
    run_id=params.fingerprint(), generated_for_as_of=params.as_of_date,
    params_json=json.dumps(pj, sort_keys=True), generator="notebook 02_load_erp_wms_sql")]))

## Summary

In [ ]:
ORDER = ["ref.market", "ref.person", "ref.decision_owner", "ref.synthetic_run", "erp.location", "erp.product_style",
         "erp.product_sku", "erp.price", "erp.vendor", "erp.wholesale_partner", "erp.wholesale_order",
         "erp.wholesale_order_line", "erp.purchase_order", "erp.purchase_order_line", "erp.margin_guardrail",
         "erp.regional_event", "wms.inventory_position", "wms.transfer_lane", "wms.inventory_movement",
         "ref.store_manager"]
assert set(ORDER) == set(TABLES) == set(PRIMARY_KEYS), set(TABLES) ^ set(ORDER)
tables = {k: TABLES[k] for k in ORDER}
for name, df in tables.items():
    assert not df.duplicated(PRIMARY_KEYS[name]).any(), f"duplicate primary key in {name}"
    assert (df["record_label"] == SYNTHETIC_LABEL).all(), name
    print(f"{name:32s} {len(df):>9,} rows")

## Load into Azure SQL (`write` mode only)

In [ ]:
if MODE == "write":
    import re
    server, database = dbutils.widgets.get("sql_server"), dbutils.widgets.get("sql_database")
    assert server, "set the sql_server widget"
    cred = dbutils.credentials.getServiceCredentialsProvider(dbutils.widgets.get("service_credential"))
    token = cred.get_token("https://database.windows.net/.default").token
    url = (f"jdbc:sqlserver://{server}:1433;database={database};encrypt=true;trustServerCertificate=false;"
           "hostNameInCertificate=*.database.windows.net;loginTimeout=60")
    jvm = spark._jvm

    def connect():
        props = jvm.java.util.Properties()
        props.setProperty("accessToken", token)
        return jvm.java.sql.DriverManager.getConnection(url, props)

    def run_sql(sql):
        """Run T-SQL split on GO lines; returns the result sets as lists of dicts."""
        out, conn = [], connect()
        try:
            st = conn.createStatement()
            for batch in [b.strip() for b in re.split(r"^\s*GO\s*$", sql, flags=re.M) if b.strip()]:
                has_rs = st.execute(batch)
                while True:
                    if has_rs:
                        rs = st.getResultSet(); m = rs.getMetaData(); n = m.getColumnCount()
                        cols = [m.getColumnLabel(k + 1) for k in range(n)]
                        rows = []
                        while rs.next():
                            rows.append({cols[k]: (None if rs.getObject(k + 1) is None else str(rs.getObject(k + 1))) for k in range(n)})
                        out.append(rows)
                    elif st.getUpdateCount() == -1:
                        break
                    has_rs = st.getMoreResults()
        finally:
            conn.close()
        return out

    def scalar(q):
        conn = connect()
        try:
            rs = conn.createStatement().executeQuery(q); rs.next(); return rs.getObject(1)
        finally:
            conn.close()

    print("connected as:", scalar("SELECT SUSER_SNAME()"), "| server:", scalar("SELECT @@SERVERNAME"))
else:
    print("preview mode: nothing written")


### Create the ERP/WMS tables (`write` mode only)

Idempotent: every table is created only if it is missing. Every table has a primary key (Fabric mirroring needs one) and a `record_label = 'SYNTHETIC'` check.

In [ ]:
ERP_WMS_DDL = r'''
-- Every table has a primary key (required for Fabric mirroring) and a record_label = 'SYNTHETIC' check.

IF SCHEMA_ID(N'ref') IS NULL EXEC(N'CREATE SCHEMA [ref]');
IF SCHEMA_ID(N'erp') IS NULL EXEC(N'CREATE SCHEMA [erp]');
IF SCHEMA_ID(N'wms') IS NULL EXEC(N'CREATE SCHEMA [wms]');
GO
IF OBJECT_ID(N'ref.market') IS NULL
CREATE TABLE [ref].[market] (
    [market_id] NVARCHAR(16) NOT NULL,
    [market_name] NVARCHAR(32) NULL,
    [region] NVARCHAR(32) NULL,
    [currency] NVARCHAR(16) NULL,
    [fx_usd_to_local] DECIMAL(19,4) NULL,
    [price_premium] FLOAT NULL,
    [hemisphere] NVARCHAR(16) NULL,
    [timezone] NVARCHAR(32) NULL,
    [price_rounding] NVARCHAR(16) NULL,
    [record_label] NVARCHAR(16) NOT NULL,
    CONSTRAINT [pk_ref_market] PRIMARY KEY ([market_id]),
    CONSTRAINT [ck_ref_market_synthetic] CHECK ([record_label] = N'SYNTHETIC')
);
GO
IF OBJECT_ID(N'ref.person') IS NULL
CREATE TABLE [ref].[person] (
    [person_id] NVARCHAR(16) NOT NULL,
    [display_name] NVARCHAR(32) NULL,
    [title] NVARCHAR(112) NULL,
    [home_market_id] NVARCHAR(16) NULL,
    [email] NVARCHAR(48) NULL,
    [record_label] NVARCHAR(16) NOT NULL,
    CONSTRAINT [pk_ref_person] PRIMARY KEY ([person_id]),
    CONSTRAINT [ck_ref_person_synthetic] CHECK ([record_label] = N'SYNTHETIC')
);
GO
IF OBJECT_ID(N'ref.decision_owner') IS NULL
CREATE TABLE [ref].[decision_owner] (
    [owner_row_id] INT NOT NULL,
    [category] NVARCHAR(16) NULL,
    [market_id] NVARCHAR(16) NULL,
    [role] NVARCHAR(32) NULL,
    [person_id] NVARCHAR(16) NULL,
    [approval_limit_units] INT NULL,
    [record_label] NVARCHAR(16) NOT NULL,
    CONSTRAINT [pk_ref_decision_owner] PRIMARY KEY ([owner_row_id]),
    CONSTRAINT [ck_ref_decision_owner_synthetic] CHECK ([record_label] = N'SYNTHETIC')
);
GO
IF OBJECT_ID(N'ref.synthetic_run') IS NULL
CREATE TABLE [ref].[synthetic_run] (
    [run_id] NVARCHAR(32) NOT NULL,
    [generated_for_as_of] DATE NULL,
    [params_json] NVARCHAR(MAX) NULL,
    [generator] NVARCHAR(32) NULL,
    [record_label] NVARCHAR(16) NOT NULL,
    CONSTRAINT [pk_ref_synthetic_run] PRIMARY KEY ([run_id]),
    CONSTRAINT [ck_ref_synthetic_run_synthetic] CHECK ([record_label] = N'SYNTHETIC')
);
GO
IF OBJECT_ID(N'erp.location') IS NULL
CREATE TABLE [erp].[location] (
    [location_id] NVARCHAR(16) NOT NULL,
    [location_name] NVARCHAR(80) NULL,
    [location_type] NVARCHAR(16) NULL,
    [format] NVARCHAR(16) NULL,
    [market_id] NVARCHAR(16) NULL,
    [city] NVARCHAR(32) NULL,
    [state_province] NVARCHAR(48) NULL,
    [latitude] FLOAT NULL,
    [longitude] FLOAT NULL,
    [timezone] NVARCHAR(32) NULL,
    [serving_dc_id] NVARCHAR(16) NULL,
    [opened_date] DATE NULL,
    [record_label] NVARCHAR(16) NOT NULL,
    CONSTRAINT [pk_erp_location] PRIMARY KEY ([location_id]),
    CONSTRAINT [ck_erp_location_synthetic] CHECK ([record_label] = N'SYNTHETIC')
);
GO
IF OBJECT_ID(N'erp.product_style') IS NULL
CREATE TABLE [erp].[product_style] (
    [style_id] NVARCHAR(16) NOT NULL,
    [style_name] NVARCHAR(64) NULL,
    [model] NVARCHAR(32) NULL,
    [category] NVARCHAR(16) NULL,
    [division] NVARCHAR(16) NULL,
    [gender] NVARCHAR(16) NULL,
    [size_run] NVARCHAR(16) NULL,
    [msrp_usd] DECIMAL(19,4) NULL,
    [launch_date] DATE NULL,
    [lifecycle] NVARCHAR(16) NULL,
    [unit_cost_usd] DECIMAL(19,4) NULL,
    [record_label] NVARCHAR(16) NOT NULL,
    CONSTRAINT [pk_erp_product_style] PRIMARY KEY ([style_id]),
    CONSTRAINT [ck_erp_product_style_synthetic] CHECK ([record_label] = N'SYNTHETIC')
);
GO
IF OBJECT_ID(N'erp.product_sku') IS NULL
CREATE TABLE [erp].[product_sku] (
    [sku_id] NVARCHAR(32) NOT NULL,
    [style_id] NVARCHAR(16) NULL,
    [color_code] NVARCHAR(16) NULL,
    [color_name] NVARCHAR(48) NULL,
    [size] NVARCHAR(16) NULL,
    [size_run] NVARCHAR(16) NULL,
    [gtin] NVARCHAR(32) NULL,
    [record_label] NVARCHAR(16) NOT NULL,
    CONSTRAINT [pk_erp_product_sku] PRIMARY KEY ([sku_id]),
    CONSTRAINT [ck_erp_product_sku_synthetic] CHECK ([record_label] = N'SYNTHETIC')
);
GO
IF OBJECT_ID(N'erp.price') IS NULL
CREATE TABLE [erp].[price] (
    [style_id] NVARCHAR(16) NOT NULL,
    [market_id] NVARCHAR(16) NOT NULL,
    [currency] NVARCHAR(16) NULL,
    [full_price_local] DECIMAL(19,4) NULL,
    [markdown_pct] DECIMAL(19,4) NULL,
    [effective_from] DATE NULL,
    [record_label] NVARCHAR(16) NOT NULL,
    CONSTRAINT [pk_erp_price] PRIMARY KEY ([style_id], [market_id]),
    CONSTRAINT [ck_erp_price_synthetic] CHECK ([record_label] = N'SYNTHETIC')
);
GO
IF OBJECT_ID(N'erp.vendor') IS NULL
CREATE TABLE [erp].[vendor] (
    [vendor_id] NVARCHAR(16) NOT NULL,
    [vendor_name] NVARCHAR(80) NULL,
    [country] NVARCHAR(16) NULL,
    [lead_days] INT NULL,
    [record_label] NVARCHAR(16) NOT NULL,
    CONSTRAINT [pk_erp_vendor] PRIMARY KEY ([vendor_id]),
    CONSTRAINT [ck_erp_vendor_synthetic] CHECK ([record_label] = N'SYNTHETIC')
);
GO
IF OBJECT_ID(N'erp.wholesale_partner') IS NULL
CREATE TABLE [erp].[wholesale_partner] (
    [partner_id] NVARCHAR(16) NOT NULL,
    [partner_name] NVARCHAR(32) NULL,
    [market_id] NVARCHAR(16) NULL,
    [tier] NVARCHAR(16) NULL,
    [ship_from_dc_id] NVARCHAR(16) NULL,
    [record_label] NVARCHAR(16) NOT NULL,
    CONSTRAINT [pk_erp_wholesale_partner] PRIMARY KEY ([partner_id]),
    CONSTRAINT [ck_erp_wholesale_partner_synthetic] CHECK ([record_label] = N'SYNTHETIC')
);
GO
IF OBJECT_ID(N'erp.wholesale_order') IS NULL
CREATE TABLE [erp].[wholesale_order] (
    [order_id] NVARCHAR(16) NOT NULL,
    [partner_id] NVARCHAR(16) NULL,
    [market_id] NVARCHAR(16) NULL,
    [ship_from_location_id] NVARCHAR(16) NULL,
    [order_date] DATE NULL,
    [requested_ship_date] DATE NULL,
    [order_status] NVARCHAR(16) NULL,
    [currency] NVARCHAR(16) NULL,
    [record_label] NVARCHAR(16) NOT NULL,
    CONSTRAINT [pk_erp_wholesale_order] PRIMARY KEY ([order_id]),
    CONSTRAINT [ck_erp_wholesale_order_synthetic] CHECK ([record_label] = N'SYNTHETIC')
);
GO
IF OBJECT_ID(N'erp.wholesale_order_line') IS NULL
CREATE TABLE [erp].[wholesale_order_line] (
    [order_id] NVARCHAR(16) NOT NULL,
    [line_no] INT NOT NULL,
    [sku_id] NVARCHAR(32) NULL,
    [ordered_units] INT NULL,
    [line_status] NVARCHAR(16) NULL,
    [source_message_id] NVARCHAR(48) NULL,
    [record_label] NVARCHAR(16) NOT NULL,
    CONSTRAINT [pk_erp_wholesale_order_line] PRIMARY KEY ([order_id], [line_no]),
    CONSTRAINT [ck_erp_wholesale_order_line_synthetic] CHECK ([record_label] = N'SYNTHETIC')
);
GO
IF OBJECT_ID(N'erp.purchase_order') IS NULL
CREATE TABLE [erp].[purchase_order] (
    [po_id] NVARCHAR(16) NOT NULL,
    [vendor_id] NVARCHAR(16) NULL,
    [ship_to_location_id] NVARCHAR(16) NULL,
    [style_id] NVARCHAR(16) NULL,
    [po_date] DATE NULL,
    [original_promise_date] DATE NULL,
    [current_promise_date] DATE NULL,
    [po_status] NVARCHAR(16) NULL,
    [incoterm] NVARCHAR(16) NULL,
    [record_label] NVARCHAR(16) NOT NULL,
    CONSTRAINT [pk_erp_purchase_order] PRIMARY KEY ([po_id]),
    CONSTRAINT [ck_erp_purchase_order_synthetic] CHECK ([record_label] = N'SYNTHETIC')
);
GO
IF OBJECT_ID(N'erp.purchase_order_line') IS NULL
CREATE TABLE [erp].[purchase_order_line] (
    [po_id] NVARCHAR(16) NOT NULL,
    [line_no] INT NOT NULL,
    [sku_id] NVARCHAR(32) NULL,
    [ordered_units] INT NULL,
    [unit_cost_usd] DECIMAL(19,4) NULL,
    [record_label] NVARCHAR(16) NOT NULL,
    CONSTRAINT [pk_erp_purchase_order_line] PRIMARY KEY ([po_id], [line_no]),
    CONSTRAINT [ck_erp_purchase_order_line_synthetic] CHECK ([record_label] = N'SYNTHETIC')
);
GO
IF OBJECT_ID(N'erp.margin_guardrail') IS NULL
CREATE TABLE [erp].[margin_guardrail] (
    [market_id] NVARCHAR(16) NOT NULL,
    [category] NVARCHAR(16) NOT NULL,
    [lifecycle] NVARCHAR(16) NOT NULL,
    [min_gross_margin_pct] DECIMAL(19,4) NULL,
    [protection_days] INT NULL,
    [max_transfer_cost_pct] DECIMAL(19,4) NULL,
    [record_label] NVARCHAR(16) NOT NULL,
    CONSTRAINT [pk_erp_margin_guardrail] PRIMARY KEY ([market_id], [category], [lifecycle]),
    CONSTRAINT [ck_erp_margin_guardrail_synthetic] CHECK ([record_label] = N'SYNTHETIC')
);
GO
IF OBJECT_ID(N'erp.regional_event') IS NULL
CREATE TABLE [erp].[regional_event] (
    [event_id] NVARCHAR(32) NOT NULL,
    [market_id] NVARCHAR(16) NULL,
    [event_name] NVARCHAR(48) NULL,
    [start_date] DATE NULL,
    [end_date] DATE NULL,
    [categories] NVARCHAR(48) NULL,
    [demand_uplift] FLOAT NULL,
    [record_label] NVARCHAR(16) NOT NULL,
    CONSTRAINT [pk_erp_regional_event] PRIMARY KEY ([event_id]),
    CONSTRAINT [ck_erp_regional_event_synthetic] CHECK ([record_label] = N'SYNTHETIC')
);
GO
IF OBJECT_ID(N'wms.inventory_position') IS NULL
CREATE TABLE [wms].[inventory_position] (
    [location_id] NVARCHAR(16) NOT NULL,
    [sku_id] NVARCHAR(32) NOT NULL,
    [on_hand_units] INT NULL,
    [allocated_units] INT NULL,
    [reserved_units] INT NULL,
    [in_transit_units] INT NULL,
    [snapshot_ts] DATETIME2(0) NULL,
    [inventory_status] NVARCHAR(16) NULL,
    [record_label] NVARCHAR(16) NOT NULL,
    CONSTRAINT [pk_wms_inventory_position] PRIMARY KEY ([location_id], [sku_id]),
    CONSTRAINT [ck_wms_inventory_position_synthetic] CHECK ([record_label] = N'SYNTHETIC')
);
GO
IF OBJECT_ID(N'wms.transfer_lane') IS NULL
CREATE TABLE [wms].[transfer_lane] (
    [from_location_id] NVARCHAR(16) NOT NULL,
    [to_location_id] NVARCHAR(16) NOT NULL,
    [mode] NVARCHAR(16) NULL,
    [lead_days] INT NULL,
    [cost_per_unit_usd] DECIMAL(19,4) NULL,
    [record_label] NVARCHAR(16) NOT NULL,
    CONSTRAINT [pk_wms_transfer_lane] PRIMARY KEY ([from_location_id], [to_location_id]),
    CONSTRAINT [ck_wms_transfer_lane_synthetic] CHECK ([record_label] = N'SYNTHETIC')
);
GO
IF OBJECT_ID(N'wms.inventory_movement') IS NULL
CREATE TABLE [wms].[inventory_movement] (
    [movement_id] NVARCHAR(16) NOT NULL,
    [movement_ts] DATETIME2(0) NULL,
    [movement_type] NVARCHAR(16) NULL,
    [sku_id] NVARCHAR(32) NULL,
    [from_location_id] NVARCHAR(16) NULL,
    [to_location_id] NVARCHAR(16) NULL,
    [units] INT NULL,
    [reference_id] NVARCHAR(32) NULL,
    [record_label] NVARCHAR(16) NOT NULL,
    CONSTRAINT [pk_wms_inventory_movement] PRIMARY KEY ([movement_id]),
    CONSTRAINT [ck_wms_inventory_movement_synthetic] CHECK ([record_label] = N'SYNTHETIC')
);
GO
IF OBJECT_ID(N'ref.store_manager') IS NULL
CREATE TABLE [ref].[store_manager] (
    [location_id] NVARCHAR(16) NOT NULL,
    [person_id] NVARCHAR(16) NULL,
    [record_label] NVARCHAR(16) NOT NULL,
    CONSTRAINT [pk_ref_store_manager] PRIMARY KEY ([location_id]),
    CONSTRAINT [ck_ref_store_manager_synthetic] CHECK ([record_label] = N'SYNTHETIC')
);
GO
'''
if MODE == "write":
    run_sql(ERP_WMS_DDL)
    print("ERP/WMS tables ready")


### Create the write-back tables (`write` mode only)

`erp.allocation_decision` and `erp.stock_transfer_order` are written by the coworker agent only after a named person approves in Teams. This notebook never deletes the agent's rows; it only replaces the seeded history below.

In [ ]:
WRITEBACK_DDL = r'''
-- Write-back targets for the coworker agent (SYNTHETIC demo).
-- A rebalancing proposal is written ONLY after a named human approves it; the agent identity gets
-- INSERT on these two tables and nothing else in erp.*  (see docs/architecture.md, "Write-back").

IF OBJECT_ID(N'erp.allocation_decision') IS NULL
CREATE TABLE [erp].[allocation_decision] (
    [decision_id]          UNIQUEIDENTIFIER NOT NULL DEFAULT NEWID(),
    [proposal_id]          NVARCHAR(64)  NOT NULL,
    [style_id]             NVARCHAR(16)  NOT NULL,
    [market_id]            NVARCHAR(8)   NOT NULL,
    [decision]             NVARCHAR(16)  NOT NULL CONSTRAINT [ck_erp_allocation_decision_decision]
                               CHECK ([decision] IN (N'APPROVED', N'REJECTED', N'MODIFIED')),
    [approved_by_upn]      NVARCHAR(256) NOT NULL,
    [approver_person_id]   NVARCHAR(32)  NULL,
    [rationale]            NVARCHAR(2000) NULL,
    [evidence_json]        NVARCHAR(MAX) NULL,
    [decided_at_utc]       DATETIME2(0)  NOT NULL DEFAULT SYSUTCDATETIME(),
    [record_label]         NVARCHAR(16)  NOT NULL DEFAULT N'SYNTHETIC',
    CONSTRAINT [pk_erp_allocation_decision] PRIMARY KEY ([decision_id]),
    CONSTRAINT [ck_erp_allocation_decision_synthetic] CHECK ([record_label] = N'SYNTHETIC')
);
GO
IF OBJECT_ID(N'erp.stock_transfer_order') IS NULL
CREATE TABLE [erp].[stock_transfer_order] (
    [sto_id]               NVARCHAR(32)  NOT NULL,
    [line_no]              INT           NOT NULL,
    [decision_id]          UNIQUEIDENTIFIER NOT NULL,
    [sku_id]               NVARCHAR(48)  NOT NULL,
    [from_location_id]     NVARCHAR(32)  NOT NULL,
    [to_location_id]       NVARCHAR(32)  NOT NULL,
    [units]                INT           NOT NULL CONSTRAINT [ck_erp_sto_units] CHECK ([units] > 0),
    [sto_status]           NVARCHAR(16)  NOT NULL DEFAULT N'CREATED',
    [requested_ship_date]  DATE          NULL,
    [created_at_utc]       DATETIME2(0)  NOT NULL DEFAULT SYSUTCDATETIME(),
    [record_label]         NVARCHAR(16)  NOT NULL DEFAULT N'SYNTHETIC',
    CONSTRAINT [pk_erp_stock_transfer_order] PRIMARY KEY ([sto_id], [line_no]),
    CONSTRAINT [fk_erp_sto_decision] FOREIGN KEY ([decision_id]) REFERENCES [erp].[allocation_decision]([decision_id]),
    CONSTRAINT [ck_erp_stock_transfer_order_synthetic] CHECK ([record_label] = N'SYNTHETIC')
);
GO
'''
if MODE == "write":
    run_sql(WRITEBACK_DDL)
    print("write-back tables ready")


In [ ]:
results = {}
if MODE == "write":
    spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "true")
    conn = connect()
    try:
        st = conn.createStatement()
        for name in reversed(list(tables)):
            st.execute(f"DELETE FROM {name}")
    finally:
        conn.close()
    for name, pdf in tables.items():
        t0 = time.time()
        pdf = pdf.copy()
        for c in pdf.columns:
            if isinstance(pdf[c].dtype, pd.DatetimeTZDtype):
                pdf[c] = pdf[c].dt.tz_localize(None)
            if pdf[c].isna().all():
                pdf[c] = pdf[c].astype("string")
        (spark.createDataFrame(pdf).repartition(4).write.format("jdbc")
            .option("url", url).option("dbtable", name).option("accessToken", token)
            .option("batchsize", 10000).option("tableLock", "false").mode("append").save())
        n = scalar(f"SELECT COUNT(*) FROM {name}")
        results[name] = n
        print(f"{name:32s} {n:>9,} rows  {time.time()-t0:5.1f}s  {'ok' if n == len(pdf) else 'MISMATCH'}")
        assert n == len(pdf), name
else:
    results = {k: len(v) for k, v in tables.items()}
    display(tables["wms.inventory_position"].head(20))

### `erp.allocation_decision` + `erp.stock_transfer_order`: prior decision history (DS-10, `write` mode only)

Six synthetic decisions from Sep–Oct 2026 (3 approved, 1 modified, 2 rejected) and the transfer orders behind them. Idempotent: rows with `proposal_id` `HIST-%` and `sto_id` `STO-H%` are replaced; decisions written later by the agent are untouched. Gold parses `evidence_json` into the `decision_*` tables.

In [ ]:
DECISION_HISTORY_SQL = r'''
-- SYNTHETIC prior allocation decisions (DS-10 "ERP history") for the Kinetiq demo. Idempotent: rows with
-- proposal_id 'HIST-%' / sto_id 'STO-H%' are replaced; decisions written later by the agent are untouched.
-- Keys match the generated ERP/WMS data (launch styles HM9443, FV8376, HQ9608; DC lanes; accountable owners in
-- ref.decision_owner) and all decisions predate the synthetic as-of date 2026-10-31.
-- evidence_json carries the case, the options the approver saw, and the measured outcome; Gold parses it into
-- gold.decision_case / decision_recommendation / decision_record / decision_action / decision_outcome.

SET XACT_ABORT ON;
BEGIN TRANSACTION;

DELETE FROM [erp].[stock_transfer_order] WHERE [sto_id] LIKE N'STO-H%';
DELETE FROM [erp].[allocation_decision] WHERE [proposal_id] LIKE N'HIST-%';

INSERT INTO [erp].[allocation_decision]
    ([decision_id], [proposal_id], [style_id], [market_id], [decision], [approved_by_upn], [approver_person_id],
     [rationale], [evidence_json], [decided_at_utc], [record_label])
VALUES
(N'5e1d0c01-0000-4000-8000-000000000001', N'HIST-0001', N'HM9443', N'US', N'APPROVED',
 N'dortiz@kinetiq.example', N'dortiz',
 N'Week-1 sell-through on AeroGlide 5 sizes 9-10 is 2.4x plan in the US; DE has 9 weeks of cover on the same sizes. Approved the DE to US air transfer.',
 N'{"case":{"trigger_id":"T1","sku_id":"HM9443-001-9.5","severity":"HIGH","opened_at":"2026-09-14T08:10:00Z","summary":"US stock-out risk on HM9443 Black/White sizes 9-10 eight days after launch; DE surplus on the same sizes."},
   "reason_code":"SUSTAINED_DEMAND","chosen_option_id":"S1","approved_units":120,
   "options":[{"option_id":"S0","option_label":"Maintain allocation","units":0,"projected_gm_usd":3960.0,"transfer_cost_usd":0.0,"confidence":0.70},
              {"option_id":"S1","option_label":"Rebalance markets: DE to US (air, 7 days)","units":120,"source_location_id":"DC-DE-ERF","destination_location_id":"DC-US-ALT","projected_gm_usd":10304.4,"transfer_cost_usd":447.6,"confidence":0.78},
              {"option_id":"S2","option_label":"Rebalance + expedite PO","units":220,"source_location_id":"DC-DE-ERF","destination_location_id":"DC-US-ALT","projected_gm_usd":11050.0,"transfer_cost_usd":1890.0,"confidence":0.55}],
   "outcome":{"measured_at":"2026-09-28T00:00:00Z","horizon_days":14,"metric":"full_price_units_sold","projected":120,"actual":108}}',
 '2026-09-14T13:40:00', N'SYNTHETIC'),

(N'5e1d0c01-0000-4000-8000-000000000002', N'HIST-0002', N'FV8376', N'JP', N'REJECTED',
 N'mtan@kinetiq.example', N'mtan',
 N'Intent jump in JP follows the Tokyo trail-race weekend; it fell back to baseline within three days. Not sustained, so no transfer from AU.',
 N'{"case":{"trigger_id":"T1","sku_id":"FV8376-700-8","severity":"MEDIUM","opened_at":"2026-09-22T06:30:00Z","summary":"JP intent index 2.1x peers on Summit Trek 3 Laser Orange; cover below 2 weeks on size 8."},
   "reason_code":"DEMAND_NOT_SUSTAINED","chosen_option_id":"S0","approved_units":0,
   "options":[{"option_id":"S0","option_label":"Maintain allocation","units":0,"projected_gm_usd":4700.0,"transfer_cost_usd":0.0,"confidence":0.62},
              {"option_id":"S1","option_label":"Rebalance markets: AU to JP (air, 8 days)","units":60,"source_location_id":"DC-AU-TRU","destination_location_id":"DC-JP-ICH","projected_gm_usd":5085.0,"transfer_cost_usd":291.0,"confidence":0.48}],
   "recommended_option_id":"S1"}',
 '2026-09-22T10:15:00', N'SYNTHETIC'),

(N'5e1d0c01-0000-4000-8000-000000000003', N'HIST-0003', N'HQ9608', N'DE', N'MODIFIED',
 N'fbrandt@kinetiq.example', N'fbrandt',
 N'Approved GB to DE for Cortex Low 9 Glacier Blue but cut 200 to 140 units: 60 units stay in GB for the wholesale partner holiday allocation (confirmed by Elena Rossi).',
 N'{"case":{"trigger_id":"T3","sku_id":"HQ9608-402-10","severity":"MEDIUM","opened_at":"2026-09-29T07:45:00Z","summary":"GB aging (11 weeks of cover, 22% full-price sell-through) while DE runs out in 9 days on HQ9608 Glacier Blue sizes 9-11."},
   "reason_code":"PARTNER_COMMITMENT","chosen_option_id":"S1","approved_units":140,
   "options":[{"option_id":"S0","option_label":"Maintain allocation (GB markdown later)","units":0,"projected_gm_usd":6900.0,"transfer_cost_usd":0.0,"confidence":0.66},
              {"option_id":"S1","option_label":"Rebalance markets: GB to DE (ocean-air, 5 days)","units":200,"source_location_id":"DC-GB-DAV","destination_location_id":"DC-DE-ERF","projected_gm_usd":14558.0,"transfer_cost_usd":162.0,"confidence":0.74}],
   "outcome":{"measured_at":"2026-10-14T00:00:00Z","horizon_days":14,"metric":"full_price_units_sold","projected":140,"actual":112}}',
 '2026-09-30T11:20:00', N'SYNTHETIC'),

(N'5e1d0c01-0000-4000-8000-000000000004', N'HIST-0004', N'HM9443', N'GB', N'APPROVED',
 N'fbrandt@kinetiq.example', N'fbrandt',
 N'White/Black is slow in GB but still inside launch price protection. Approved member-targeted visibility instead of an early markdown.',
 N'{"case":{"trigger_id":"T2","sku_id":"HM9443-100-10","severity":"LOW","opened_at":"2026-10-06T09:00:00Z","summary":"GB aging risk on HM9443 White/Black: 9.4 weeks of cover, 24% full-price sell-through since launch."},
   "reason_code":"PRICE_PROTECTION","chosen_option_id":"S4","approved_units":0,
   "options":[{"option_id":"S0","option_label":"Maintain allocation (markdown after protection window)","units":0,"projected_gm_usd":1950.0,"transfer_cost_usd":0.0,"confidence":0.60},
              {"option_id":"S4","option_label":"Targeted demand shaping: member visibility, no markdown","units":0,"projected_gm_usd":3200.0,"transfer_cost_usd":0.0,"confidence":0.64}],
   "action":{"action_type":"DEMAND_SHAPING","target_system":"CRM","status":"SUCCEEDED","executed_at":"2026-10-07T08:00:00Z"},
   "outcome":{"measured_at":"2026-10-21T00:00:00Z","horizon_days":14,"metric":"full_price_sell_through_pts","projected":4.0,"actual":3.1}}',
 '2026-10-06T16:30:00', N'SYNTHETIC'),

(N'5e1d0c01-0000-4000-8000-000000000005', N'HIST-0005', N'FV8376', N'US', N'REJECTED',
 N'dortiz@kinetiq.example', N'dortiz',
 N'Inbound PO for Summit Trek 3 lands in Ontario on Oct 18, before a BR transfer would arrive. Holding for the PO.',
 N'{"case":{"trigger_id":"T1","sku_id":"FV8376-001-7.5","severity":"MEDIUM","opened_at":"2026-10-14T12:00:00Z","summary":"US cover 1.3 weeks on FV8376 Black/White size 7.5; BR has surplus."},
   "reason_code":"INBOUND_PO_SOONER","chosen_option_id":"S0","approved_units":0,
   "options":[{"option_id":"S0","option_label":"Maintain allocation (wait for inbound PO)","units":0,"projected_gm_usd":6420.0,"transfer_cost_usd":0.0,"confidence":0.71},
              {"option_id":"S1","option_label":"Rebalance markets: BR to US (air, 8 days)","units":80,"source_location_id":"DC-BR-CAJ","destination_location_id":"DC-US-ALT","projected_gm_usd":6802.4,"transfer_cost_usd":365.6,"confidence":0.57}],
   "recommended_option_id":"S1"}',
 '2026-10-14T15:05:00', N'SYNTHETIC'),

(N'5e1d0c01-0000-4000-8000-000000000006', N'HIST-0006', N'HM9443', N'US', N'APPROVED',
 N'dortiz@kinetiq.example', N'dortiz',
 N'Second AeroGlide 5 rebalance: US demand still sustained (6 of last 7 days above threshold); GB holds surplus on sizes 9-10. Approved GB to US.',
 N'{"case":{"trigger_id":"T3","sku_id":"HM9443-001-9.5","severity":"HIGH","opened_at":"2026-10-21T07:20:00Z","summary":"Mismatch pair: US stock-out in 6 days and GB 10 weeks of cover on HM9443 Black/White sizes 9-10."},
   "reason_code":"SUSTAINED_DEMAND","chosen_option_id":"S1","approved_units":150,
   "options":[{"option_id":"S0","option_label":"Maintain allocation","units":0,"projected_gm_usd":4410.0,"transfer_cost_usd":0.0,"confidence":0.69},
              {"option_id":"S1","option_label":"Rebalance markets: GB to US (ocean-air, 7 days)","units":150,"source_location_id":"DC-GB-DAV","destination_location_id":"DC-US-ALT","projected_gm_usd":12957.0,"transfer_cost_usd":483.0,"confidence":0.81},
              {"option_id":"S3","option_label":"Restrict marketplace availability in GB","units":0,"projected_gm_usd":5200.0,"transfer_cost_usd":0.0,"confidence":0.50}],
   "outcome":{"measured_at":null,"horizon_days":14,"metric":"full_price_units_sold","projected":150,"actual":null}}',
 '2026-10-21T09:50:00', N'SYNTHETIC');

INSERT INTO [erp].[stock_transfer_order]
    ([sto_id], [line_no], [decision_id], [sku_id], [from_location_id], [to_location_id], [units], [sto_status],
     [requested_ship_date], [created_at_utc], [record_label])
VALUES
(N'STO-H0001', 1, N'5e1d0c01-0000-4000-8000-000000000001', N'HM9443-001-9',   N'DC-DE-ERF', N'DC-US-ALT', 40, N'RECEIVED', '2026-09-15', '2026-09-14T14:05:00', N'SYNTHETIC'),
(N'STO-H0001', 2, N'5e1d0c01-0000-4000-8000-000000000001', N'HM9443-001-9.5', N'DC-DE-ERF', N'DC-US-ALT', 40, N'RECEIVED', '2026-09-15', '2026-09-14T14:05:00', N'SYNTHETIC'),
(N'STO-H0001', 3, N'5e1d0c01-0000-4000-8000-000000000001', N'HM9443-001-10',  N'DC-DE-ERF', N'DC-US-ALT', 40, N'RECEIVED', '2026-09-15', '2026-09-14T14:05:00', N'SYNTHETIC'),
(N'STO-H0003', 1, N'5e1d0c01-0000-4000-8000-000000000003', N'HQ9608-402-9',   N'DC-GB-DAV', N'DC-DE-ERF', 50, N'RECEIVED', '2026-10-01', '2026-09-30T11:45:00', N'SYNTHETIC'),
(N'STO-H0003', 2, N'5e1d0c01-0000-4000-8000-000000000003', N'HQ9608-402-10',  N'DC-GB-DAV', N'DC-DE-ERF', 50, N'RECEIVED', '2026-10-01', '2026-09-30T11:45:00', N'SYNTHETIC'),
(N'STO-H0003', 3, N'5e1d0c01-0000-4000-8000-000000000003', N'HQ9608-402-11',  N'DC-GB-DAV', N'DC-DE-ERF', 40, N'RECEIVED', '2026-10-01', '2026-09-30T11:45:00', N'SYNTHETIC'),
(N'STO-H0006', 1, N'5e1d0c01-0000-4000-8000-000000000006', N'HM9443-001-9',   N'DC-GB-DAV', N'DC-US-ALT', 50, N'RECEIVED', '2026-10-22', '2026-10-21T10:10:00', N'SYNTHETIC'),
(N'STO-H0006', 2, N'5e1d0c01-0000-4000-8000-000000000006', N'HM9443-001-9.5', N'DC-GB-DAV', N'DC-US-ALT', 50, N'RECEIVED', '2026-10-22', '2026-10-21T10:10:00', N'SYNTHETIC'),
(N'STO-H0006', 3, N'5e1d0c01-0000-4000-8000-000000000006', N'HM9443-001-10',  N'DC-GB-DAV', N'DC-US-ALT', 50, N'RECEIVED', '2026-10-22', '2026-10-21T10:10:00', N'SYNTHETIC');

COMMIT TRANSACTION;
GO
SELECT d.[proposal_id], d.[style_id], d.[market_id], d.[decision], d.[approver_person_id], d.[decided_at_utc],
       COUNT(s.[line_no]) AS sto_lines, COALESCE(SUM(s.[units]), 0) AS sto_units
FROM [erp].[allocation_decision] d
LEFT JOIN [erp].[stock_transfer_order] s ON s.[decision_id] = d.[decision_id]
WHERE d.[proposal_id] LIKE N'HIST-%'
GROUP BY d.[proposal_id], d.[style_id], d.[market_id], d.[decision], d.[approver_person_id], d.[decided_at_utc]
ORDER BY d.[proposal_id];
GO
'''
if MODE == "write":
    hist = run_sql(DECISION_HISTORY_SQL)[-1]
    for r in hist:
        print(r)
    assert len(hist) == 6, f"expected 6 seeded decisions, got {len(hist)}"
    results["erp.allocation_decision (history)"] = len(hist)


## Validation queries

In [ ]:
if MODE == "write":
    checks = {
      "hero PO slip days": "SELECT DATEDIFF(day, original_promise_date, current_promise_date) FROM erp.purchase_order WHERE po_id = '4577091802'",
      "DQ-04 negative on-hand rows": "SELECT COUNT(*) FROM wms.inventory_position WHERE on_hand_units < 0",
      "DQ-09 over-committed rows": "SELECT COUNT(*) FROM wms.inventory_position WHERE allocated_units + reserved_units > on_hand_units AND on_hand_units >= 0",
      "DQ-11 re-sent cancelled lines": "SELECT COUNT(*) FROM erp.wholesale_order_line WHERE source_message_id LIKE '%-RESEND'",
      "non-synthetic rows (must be 0)": "SELECT COUNT(*) FROM wms.inventory_position WHERE record_label <> 'SYNTHETIC'",
      "tables without PK (must be 0)": """SELECT COUNT(*) FROM sys.tables t WHERE SCHEMA_NAME(t.schema_id) IN ('ref','erp','wms')
                                          AND NOT EXISTS (SELECT 1 FROM sys.key_constraints k WHERE k.parent_object_id = t.object_id AND k.type = 'PK')""",
    }
    for label, q in checks.items():
        print(f"{label:34s} {scalar(q)}")
dbutils.notebook.exit(json.dumps({"mode": MODE, "fingerprint": params.fingerprint(), **results}))